# SQL: Advanced Topics

## Introduction and Table of Contents

Thus far, we have kept our discussion of what SQL actually *is* relatively brief. This has been intentional; our goal has been to get you comfortable writing SQL queries as quickly as possible so that you can use it where necessary.

In this section, we will take a moment to talk about SQL more generally and explore some more advanced topics. By the end of the notebook, you will be able to:

- Discuss SQLite interfacing commands and capabilities. Additionally, using sqlite_master, PRAGMA, and other commands to view the schema of the tables. (`Exercise 0`)
- Spend some time reviewing more advanced SQL functionality, such as common-table expressions (CTEs) and window functions. (`Exercises 1-7`)
- Discuss query logic and refactoring best practices for optimal performance
- Review some of the security implications related to working with SQL and revisit the concept of placeholder bindings. (`Exercises 8-9`)
- Showcase how you can use some of these tools to answer a frequently-asked but deceptively tricky data analysis question: the "top-n cases by group" problem. (`Exercise 10`)

Table of Contents
- [Dataset Overview: The NYC Jobs Postings Database](#Dataset-Overview:-The-NYC-Jobs-Postings-Database)
- [SQLite Interfacing](#SQLite-Interfacing)
    - [Connections and Cursors](#Connections-and-Cursors)
    - [Tables, Commands, Executing, and Committing](#Tables,-Commands,-Executing,-and-Committing)
    - [Closing Connections](#Closing-Connections)
    - [sqlite_master](#sqlite_master)
    - [PRAGMA](#PRAGMA)
- [Advanced Functionality](#Advanced-Functionality)
    - [Introduction to SQLite String Manipulation](#Introduction-to-SQLite-String-Manipulation)
    - [Introduction to CASE Statements in SQLite](#Introduction-to-CASE-Statements-in-SQLite)
    - [Handling NULL Values](#Handling-NULL-Values)
    - [Introduction to Inline Subqueries and Common Table Expressions (CTEs)](#Introduction-to-Inline-Subqueries-and-Common-Table-Expressions-&#40;CTEs&#41;)
    - [Introduction to Window Functions](#Introduction-to-Window-Functions)
    - [Handling NULL Values continued](#Handling-NULL-Values-continued)
- [Query Logic and Refactoring](#Query-Logic-and-Refactoring)
    - [Writing queries for better performance](#Writing-queries-for-better-performance)
    - [EXCEPT vs NOT IN vs LEFT JOIN](#EXCEPT-vs-NOT-IN-vs-LEFT-JOIN)
- [Security Implications and Placeholder Bindings](#Security-Implications-and-Placeholder-Bindings)
    - [Queries and User Input](#Queries-and-User-Input)
    - [SQL Injection Attacks](#SQL-Injection-Attacks)
    - [Protecting Against SQL Injections](#Protecting-Against-SQL-Injections)
- [An Applied Example: Finding the Top N Cases per Group](#An-Applied-Example:-Finding-the-Top-N-Cases-per-Group)

## Dataset Overview: The NYC Jobs Postings Database

This dataset is a normalized version of the NYC Jobs Postings data. It is available on [NYC OpenData](https://opendata.cityofnewyork.us/), which is a platform for free public data published by New York City agencies and other partners. The original dataset was retrieved on 08-30-2024. The most current version of the dataset can be found [here](https://data.cityofnewyork.us/City-Government/Jobs-NYC-Postings/kpav-sd4t/about_data).

The data contains information on current job postings available on the City of New York's official jobs site. It includes both internal postings for city employees and external postings available to the general public.

The dataset has been normalized and loaded into a SQLite database `nyc_jobs.db` to enable retieval with SQL. Below is a list of the tables included in the database along with a brief description of each:

- **job_postings**: Contains core information about each job posting, such as the job ID, title, posting date, and the date the posting expires (post_until). This is the central table linking to other detailed tables.
- **agencies**: Stores details about the agencies responsible for the job postings, including the agency ID and agency name.
- **job_titles**: Provides detailed information on the job titles listed in the postings, including the title classification and civil service title.
- **salaries**: Contains salary information for the job postings, such as the salary range and salary frequency (e.g., annual, hourly).
- **locations**: Details the locations associated with the job postings, including the specific work location and any relevant divisions or units.
- **jobs**: Provides a mechanism to join all of the previous tables together. This contains all job level information and metadata.

The image below shows the schema of these tables.

![schema](resource/asnlib/publicdata/schema_diagram.png)

Finally, we will be following the [SQL Style Guide by Simon Holywell](https://www.sqlstyle.guide/) for any SQL that we write in this notebook. Your code will run without following the guide, so it is up to you if you would like to use it.

**IMPORTANT!** Please run the cells below to properly configure your notebook environment.

In [2]:
### Global imports
import dill
from cse6040_devkit import plugins, utils
from cse6040_devkit.training_wheels import run_with_timeout, suppress_stdout
import tracemalloc
from time import time
import re 
import pandas as pd

utils.add_from_file('sql_validator', plugins)
utils.add_from_file('malicious_executor', plugins)
utils.add_from_file('sql_to_df_plugin', plugins)
utils.add_from_file('safe_executor', plugins)


cse6040_devkit.plugins
cse6040_devkit.plugins
cse6040_devkit.plugins
cse6040_devkit.plugins


In [3]:
import sqlite3
# Database connection
conn = sqlite3.connect('resource/asnlib/publicdata/nyc_jobs.db')

## SQLite Interfacing
In this section, we will briefly discuss basic SQLite interfacing. While we will exclusively use Pandas `read_sql` function throughout this notebook, the information below is good foundational knowledge.

### Connections and Cursors
The basics of creating a connection were addressed in Part 0, but this section will go into greater detail. Remember, the SQLite engine in Python maintains a database as a file; in this example, the name of that file is nyc_jobs.db.

> **Important usage note!** If the named file does **not** yet exist, this code creates it. However, if the database has been created before, this same code will open it. This fact can be important when you are debugging. For example, if your code depends on the database not existing initially, then you may need to remove the file first. You're likely to encounter this situation in this notebook.

You issue commands to the database through an object called a cursor. The cursor tracks the current state of the database, and you will mostly be using the cursor to issue commands that modify or query the database.

```python
import sqlite3

# Connect to (or create) a database
conn = sqlite3.connect('resource/asnlib/publicdata/nyc_jobs.db')

# Create a cursor object to interact with the database
c = conn.cursor()
```

### Tables, Commands, Executing, and Committing
The central object of a relational database is a table, which contains observations as rows and variables as columns. In the relational database world, we sometimes refer to rows as items or records and columns as attributes. We'll use all of these terms interchangeably in this course.

Suppose we wish to maintain a database of Georgia Tech students, whose attributes are their names and Georgia Tech-issued ID numbers. You might start by creating a table named students to hold this data. You can create the table using the command, `CREATE TABLE`.

While we're not creating tables within this notebook, if you try to create a table that already exists, it will fail. If you are trying to create tables from scratch in a SQLite database, you may need to remove any existing sqlite.db file or destroy any existing table; you can do the latter with the SQL command, `DROP TABLE IF EXISTS tablename`.

```python
# If this is not the first time you run this cell, 
# you need to delete the existing "students" table first
c.execute("DROP TABLE IF EXISTS students")

# create a table named "students" with 2 columns: "gtid" and "name".
# the type for column "gtid" is integer and for "name" is text. 
c.execute("""
CREATE TABLE students (
    gtid         INTEGER, 
    student_name TEXT
)
""")
```

To populate the table with items, you can use the command `INSERT INTO`.

```python
c.execute("INSERT INTO students VALUES (123, 'Vuduc')")
c.execute("INSERT INTO students VALUES (456, 'Chau')")
c.execute("INSERT INTO students VALUES (381, 'Bader')")
c.execute("INSERT INTO students VALUES (991, 'Sokol')")
```
Transaction Commits. The commands above modify the database. However, these are temporary modifications and aren't actually saved to the database until committed. The way to do that is to issue a commit operation from the connection object.

> There are some subtleties related to when you actually need to commit, since the SQLite database engine does commit at certain points as discussed [here](https://stackoverflow.com/questions/13642956/commit-behavior-and-atomicity-in-python-sqlite3-module). However, it's probably simpler if you remember to include commits when you intend for them to take effect.

```python
# to commit the new table
conn.commit()
```

### Closing Connections
Finally, while it's not necessary, it is good practice to close the cursor and connection, the same way you close files.
```python
# close cursor
c.close()

# close connection
conn.close()
```

### sqlite_master

Often, databases and their schema are new to users. Therefore, when we encounter databases for the first time, we may have some of the following questions:
1. What tables are in the database?
2. What is the structure/schema of each table (columns and data types)?
3. What does the data look like in each table (data sample)?

In SQLite, the table `sqlite_master` contains the metadata or [schema](https://www.sqlite.org/schematab.html) about every table in the database, including information about tables, indexes, views, and triggers. It's an internal table that SQLite uses to keep track of the structure of the database.

Luckily, we can query the sqlite_master table to retrieve information about the database schema, including details about the tables in the database, the columns in those tables, and other objects.

Structure of sqlite_master:
The sqlite_master table has the following columns:
*    type: The type of the object (e.g., table, index, view, or trigger).
*    name: The name of the object (e.g., the name of a table, index, or view).
*    tbl_name: The name of the table to which the object belongs (relevant for indexes, views, and triggers).
*    rootpage: The page number of the root b-tree page for the object (relevant for tables and indexes).
*    sql: The SQL statement that was used to create the object (e.g., the CREATE TABLE or CREATE INDEX statement).    

In [4]:
query = """
        SELECT *
          FROM sqlite_master
         WHERE type='table'
        """
pd.read_sql(query,conn)

,type,name,tbl_name,rootpage,sql
0,table,agencies,agencies,2,CREATE TABLE agencies (\n\tagency_id INTEGER N...
1,table,job_titles,job_titles,4,CREATE TABLE job_titles (\n\tjob_title_id INTE...
2,table,locations,locations,5,CREATE TABLE locations (\n\tlocation_id INTEGE...
3,table,jobs,jobs,7,CREATE TABLE jobs (\n\tjob_id VARCHAR(100) NOT...
4,table,job_postings,job_postings,9,CREATE TABLE job_postings (\n\tposting_id INTE...
5,table,salaries,salaries,11,CREATE TABLE salaries (\n\tsalary_id INTEGER N...


### PRAGMA
In SQLite, [PRAGMA](https://www.sqlite.org/pragma.html) statements can be used to query or modify database settings and retrieve metadata. To get metadata about tables, columns, indexes, and other database objects, SQLite provides specific PRAGMA commands that allow you to extract detailed information about the database schema.

Here, we are using the `table_info` function, which returns the table structure and column information about the table whose name is passed to it.

This will return the following:
*    cid: Column ID (an integer representing the column's index).
*    name: The name of the column.
*    type: The data type of the column (e.g., INTEGER, TEXT, REAL).
*    notnull: A flag indicating whether the column has a NOT NULL constraint (1 if NOT NULL, 0 if not).
*    dflt_value: The default value for the column (if any).
*    pk: Indicates whether the column is part of the primary key (1 if yes, 0 if no).

In [5]:
query = """
        PRAGMA table_info('jobs')
        """
pd.read_sql(query,conn)

,cid,name,type,notnull,dflt_value,pk
0,0,job_id,VARCHAR(100),1,None,1
1,1,agency_id,INTEGER,1,None,0
2,2,job_title_id,INTEGER,1,None,0
3,3,location_id,INTEGER,1,None,0
4,4,work_location_1,VARCHAR(500),0,None,0
5,5,division_work_unit,VARCHAR(500),0,None,0
6,6,hours_shift,VARCHAR(500),0,None,0
7,7,recruitment_contact,TEXT,0,None,0
8,8,residency_requirement,TEXT,0,None,0
9,9,job_description,TEXT,0,None,0


Finally, recall that we can "view" the data and records by using:

```sql
SELECT *
  FROM jobs
 LIMIT 10
```

In [6]:
query = """
        SELECT * 
          FROM jobs 
         LIMIT 10
        """
pd.read_sql(query,conn)

,job_id,agency_id,job_title_id,location_id,work_location_1,division_work_unit,hours_shift,recruitment_contact,residency_requirement,job_description,minimum_qual_requirements,preferred_skills,additional_information
0,639938,1,1,1,None,PUB BLDGS/A+E/ENGINEERING,None,None,New York City Residency is not required for th...,Hours: Full-Time a 35 Hours Work Location: 30-...,1. A valid New York State License as a Profess...,Candidate should process a minimum of 10-15 ye...,None
1,630960,2,2,2,None,WSO-OGI Maintenance,None,None,New York City residency is generally required ...,IMPORTANT: Candidates Must have a Commercial P...,1. One year of full-time experience in gardeni...,Commercial Pesticide Applicator - Category 3A ...,Candidates Must have a Commercial Pesticide Ap...
2,537794,3,3,3,None,Crossroads Juvenile Center,None,None,New York City Residency is not required for th...,The Administration for Childrenas Services (AC...,1. A four year high school diploma or its educ...,None,Section 424-A of the New York Social Services ...
3,540906,4,4,4,None,Support Staff,None,None,City Residency is not required for this position,JOB RESPONSIBILITIES Specific duties will incl...,1. A baccalaureate degree from an accredited c...,None,Physical Requirements: Tasks involve the abili...
4,582773,2,5,5,"96-05 Horace Harding Expressway, 2nd floor Cor...",BWT - PROCUREMENT,35 hours per week/day,None,New York City residency is generally required ...,**IMPORTANT NOTE: Only those currently serving...,1. A baccalaureate degree from an accredited c...,"1. Excellent organizational, interpersonal, ve...",Appointments are subject to OMB approval. For ...
5,638892,5,6,6,None,Sust Policy & Legal Affairs,None,None,New York City Residency is not required for th...,Join the fight against climate change and take...,1. Admission to the New York State Bar; and ei...,"a Experience with adjudication, prosecution, o...",None
6,640278,6,7,7,None,Finance Budget,None,None,New York City residency is generally required ...,To Apply: Email your resume and cover letter t...,As of June of the Program year the prospective...,The ideal candidate will have the following: a...,None
7,597843,7,8,8,None,Facilities Management,None,None,New York City residency is generally required ...,NYC DOT seeks to hire a Director of Financial ...,1. A master's degree from an accredited colleg...,a Proven quantitative and analytic skills a Kn...,*IN ORDER TO BE CONSIDERED FOR THIS POSITION C...
8,646598,8,9,9,None,Brooklyn Property Management,None,None,NYCHA has no Residency Requirements.,1. Drive development vehicles and assist in de...,Qualification Requirements There are no formal...,None,1. Possession of a valid driver's license is r...
9,648326,9,10,10,None,Operations Central,None,None,City Residency is not required for this position,The New York County District Attorney's Office...,Qualification Requirements 1. High school grad...,None,None


### Exercise 0: (1 points)
**get_database_schema__FREE**  

**Example:** we have defined `get_database_schema__FREE` as follows:

**This is a free exercise!** 

    **Please run the test cell below to collect your FREE point!**

    The output will show the structure of the database which we will use for the following exercises.


In [7]:
### Solution - Exercise 0  
def get_database_schema__FREE(conn):
    # Retrieve the list of tables from sqlite_master
    tables_query = """
    SELECT name 
      FROM sqlite_master 
     WHERE type='table';
                   """
    tables = pd.read_sql_query(tables_query, conn)

    # Prepare DataFrame to hold combined results
    schema_info = pd.DataFrame()

    # Loop through tables to get column details
    for table in tables['name']:
        # Get column details
        pragma_table_info = f"PRAGMA table_info('{table}')"
        table_info = pd.read_sql_query(pragma_table_info, conn)
        table_info['table_name'] = table  # Add table name to the DataFrame

        # Add to main DataFrame
        schema_info = pd.concat([schema_info, table_info], ignore_index=True)

    # Select and rename relevant columns
    schema_info = schema_info[['table_name', 'name', 'type']]
    schema_info.columns = ['table', 'column', 'type']

    return schema_info

### Demo function call
df_schema = get_database_schema__FREE(conn)
display(df_schema)

,table,column,type
0,agencies,agency_id,INTEGER
1,agencies,agency_name,VARCHAR(500)
2,job_titles,job_title_id,INTEGER
3,job_titles,business_title,VARCHAR(500)
4,job_titles,civil_service_title,VARCHAR(500)
5,job_titles,title_code_no,VARCHAR(100)
6,job_titles,title_classification,VARCHAR(200)
7,job_titles,career_level,VARCHAR(200)
8,job_titles,job_category,VARCHAR(200)
9,job_titles,level,VARCHAR(100)


 


 ---
 <!-- Test Cell Boilerplate -->  
 The test cell below will always pass. Please submit to collect your free points for get_database_schema__FREE (exercise 0).
 

In [8]:
### Test Cell - Exercise 0  


print('Passed! Please submit.')

Passed! Please submit.


## Advanced Functionality
In this section, we will expand our understanding of what is possible within an SQL query. While some of these concepts are more complicated than what we have discussed thus far, these tools can be extremely powerful in the proper circumstances.  

### Introduction to SQLite String Manipulation

By now, you should be familiar with Python's ability to process and manipulate textual data. Here, we will discuss SQLite and its limitations when dealing with strings, particularly when compared to more powerful tools like Python and enterprise-grade relational databases like Oracle and SQL Server.

Common SQLite string functions:

| Function | Description |
| ---- | ---- |
| LENGTH(str) | Returns the length of a string. |
| LOWER(str) | Converts a string to lowercase. |
| UPPER(str) | Converts a string to uppercase. |
| SUBSTR(str, start, length) | Extracts a substring from a string. |
| TRIM(str) | Removes leading and trailing spaces from a string. |
| LTRIM(str) | Removes leading spaces from a string. |
| RTRIM(str) | Removes trailing spaces from a string. |
| REPLACE(str, old, new) | Replaces occurrences of a substring with a new one. |
| INSTR(str, substr) | Returns the position of the first occurrence of a substring. |
| CONCAT(str1, str2, ...) or &#124;&#124; | Concatenates strings. |
| LIKE | Performs pattern matching in strings. |
| GROUP_CONCAT(str) | Concatenates multiple rows into one string with a delimiter. |

SQLite offers a basic set of string functions, which can make complex string operations challenging. Notably, SQLite lacks:
- Built-in support for regular expressions
- Advanced string to datetime parsing functionality

These limitations can force you to implement more creative solutions, often involving combinations of functions like `TRIM`, `SUBSTR`, `INSTR`, and `REPLACE` to achieve results that would be straightforward in other environments. The following exercise will ask you to get creative within these constraints while extracting meaningful data from the NYC Jobs database.

### Exercise 1: (1 points)
**parse_classification**  

**Your task:** define `parse_classification_query` as follows:

It should query the database to create a new DataFrame from the `job_titles` table with the following columns:
- `title_classification`: The original `title_classification` column from the `job_titles` table.
- `title_classification_desc`: The text portion of the `title_classification` before the final dash (excluding the dash and integer that follows).
- `title_classification_code`: The integer portion that appears after the final dash in `title_classification`.

>While it says `integer portion` above, the SELECT portion should return string types. Do not use `CAST()`.

**Requirements/steps**:
- The database table you will need is named `job_titles`.
- The column you will work with is `title_classification`.

**Hint**: There are multiple ways to solve this exercise. The following SQLite functions may help, but not all are strictly necessary, [SUBSTR](https://www.sqlite.org/lang_corefunc.html#substr), [INSTR](https://www.sqlite.org/lang_corefunc.html#instr), and [REPLACE](https://sqlite.org/lang_corefunc.html#replace).

In [9]:
### Solution - Exercise 1  
parse_classification_query = '''
SELECT title_classification,
  RTRIM(title_classification,'0123456789-') AS title_classification_desc,
  SUBSTR(title_classification, -1) AS title_classification_code
FROM job_titles;

'''

### Demo function call
demo_result_parse_classification = pd.read_sql_query(parse_classification_query, conn)
display(demo_result_parse_classification.head())

,title_classification,title_classification_desc,title_classification_code
0,Competitive-1,Competitive,1
1,Competitive-1,Competitive,1
2,Competitive-1,Competitive,1
3,Non-Competitive-5,Non-Competitive,5
4,Competitive-1,Competitive,1


 

**The demo should display this output.**  

<div style="overflow-x:auto;">
    <style>
        table {
            border-collapse: collapse;
            margin: 15px 0;
            font-size: 0.9em;
            font-family: sans-serif;
            width: auto;
        }
        th, td {
            padding: 8px;
            text-align: right;
            border-bottom: 1px solid #ddd;
        }
        tr:hover {background-color: #f5f5f5;}
    </style>
    <table border="0" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>title_classification</th>
      <th>title_classification_desc</th>
      <th>title_classification_code</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>0</th>
      <td>Competitive-1</td>
      <td>Competitive</td>
      <td>1</td>
    </tr>
    <tr>
      <th>1</th>
      <td>Competitive-1</td>
      <td>Competitive</td>
      <td>1</td>
    </tr>
    <tr>
      <th>2</th>
      <td>Competitive-1</td>
      <td>Competitive</td>
      <td>1</td>
    </tr>
    <tr>
      <th>3</th>
      <td>Non-Competitive-5</td>
      <td>Non-Competitive</td>
      <td>5</td>
    </tr>
    <tr>
      <th>4</th>
      <td>Competitive-1</td>
      <td>Competitive</td>
      <td>1</td>
    </tr>
  </tbody>
</table>
    </div>


 ---
 <!-- Test Cell Boilerplate -->  
The cell below will test your solution for parse_classification (exercise 1). The testing variables will be available for debugging under the following names in a dictionary format.  
- `input_vars` - Input variables for your solution.   
- `original_input_vars` - Copy of input variables from prior to running your solution. Any `key:value` pair in `original_input_vars` should also exist in `input_vars` - otherwise the inputs were modified by your solution.  
- `returned_output_vars` - Outputs returned by your solution.  
- `true_output_vars` - The expected output. This _should_ "match" `returned_output_vars` based on the question requirements - otherwise, your solution is not returning the correct output. 


In [10]:
### Test Cell - Exercise 1  


from cse6040_devkit.tester_fw.testers import Tester
from yaml import safe_load
from time import time

tracemalloc.start()
mem_start, peak_start = tracemalloc.get_traced_memory()
print(f"initial memory usage: {mem_start/1024/1024:.2f} MB")

# Load testing utility
with open('resource/asnlib/publicdata/execute_tests', 'rb') as f:
    executor = dill.load(f)

@run_with_timeout(error_threshold=200.0, warning_threshold=100.0)
@suppress_stdout
def execute_tests(**kwargs):
    return executor(**kwargs)


# Execute test
start_time = time()
passed, test_case_vars, e = execute_tests(func=plugins.sql_executor(parse_classification_query),
              ex_name='parse_classification',
              key=b'btJ0h55N40azEpSYp7KWZGFYuc4IQesVgtsBQFmcS-8=', 
              n_iter=10)
# Assign test case vars for debugging
input_vars, original_input_vars, returned_output_vars, true_output_vars = test_case_vars
duration = time() - start_time
print(f"Test duration: {duration:.2f} seconds")
current_memory, peak_memory = tracemalloc.get_traced_memory()
print(f"memory after test: {current_memory/1024/1024:.2f} MB")
print(f"memory peak during test: {peak_memory/1024/1024:.2f} MB")
tracemalloc.stop()
if e: raise e
assert passed, 'The solution to parse_classification did not pass the test.'

###
### AUTOGRADER TEST - DO NOT REMOVE
###

print('Passed! Please submit.')

initial memory usage: 0.00 MB
Test duration: 0.29 seconds
memory after test: 3.16 MB
memory peak during test: 4.40 MB
Passed! Please submit.


### Introduction to CASE Statements in SQLite

In SQLite, the `CASE` statement is a powerful tool that allows you to recode variables based on specific conditions. It works similarly to an `IF-THEN-ELSE` structure in programming languages. By using `CASE`, you can create new columns, transform existing values, or recategorize data based on conditional logic directly within your SQL queries.

#### Example Case Statement

Suppose you have a table called `employees` with a column named `experience_level` that categorizes employees as `'Junior'`, `'Mid'`, or `'Senior'`. You want to create a new column called `salary_band` that assigns a salary band based on the `experience_level`. Here’s a simple example:

```sql
SELECT employee_id,
       experience_level,
       CASE
       WHEN experience_level = 'Junior' THEN 'Low'
       WHEN experience_level = 'Mid' THEN 'Medium'
       WHEN experience_level = 'Senior' THEN 'High'
       ELSE 'Unknown'
       END AS salary_band
  FROM employees
```

Let's use this gained knowledge for the following exercise.

### Exercise 2: (1 points)
**convert_post_until**  

**Your task:** define `convert_post_until_query` as follows:

It should query the `job_postings` table with the following columns in the result:
- `post_until`: The original `post_until` column from the `job_postings` table.
- `post_until_converted`: The same date as the `post_until` column converted to "YYYY-MM-DD" format.

**Requirements/steps**:
- The database table you will need is named `job_postings`.
- The column you will work with is `post_until`, which contains date strings in "DD-MMM-YYYY" format (e.g., 24-AUG-2024).
- Do not filter out `NULL` values.

**Hint**: You may want to use a [CASE](https://www.sqlitetutorial.net/sqlite-case/) statement to handle the months. [SUBSTR](https://www.sqlitetutorial.net/sqlite-functions/sqlite-substr/) and [concatenation operator](https://www.sqlitetutorial.net/sqlite-string-functions/sqlite-concat/) SQLite functions may be helpful for this exercise.

> concat() is available in SQLite version 3.44.0 and newer. You'll need to run `print(sqlite3.sqlite_version)` to get the current version of SQLite.

In [11]:
### Solution - Exercise 2  
convert_post_until_query = '''
SELECT post_until,
     SUBSTR(post_until, 8, 4) ||'-'|| --year
     CASE SUBSTR(post_until, 4, 3)
     WHEN 'JAN' THEN '01'
     WHEN 'FEB' THEN '02'
     WHEN 'MAR' THEN '03'
     WHEN 'APR' THEN '04'
     WHEN 'MAY' THEN '05'
     WHEN 'JUN' THEN '06'
     WHEN 'JUL' THEN '07'
     WHEN 'AUG' THEN '08'
     WHEN 'SEP' THEN '09'
     WHEN 'OCT' THEN '10'
     WHEN 'NOV' THEN '11'
     WHEN 'DEC' THEN '12'
     END ||'-'||
     SUBSTR(post_until, 1, 2) --day
AS post_until_converted

FROM job_postings;

'''
### Demo function call
demo_result_convert_post_until = pd.read_sql_query(convert_post_until_query, conn)
post_until_not_null = demo_result_convert_post_until['post_until'].notnull()
display(demo_result_convert_post_until[post_until_not_null].head())

,post_until,post_until_converted
3,03-JAN-2025,2025-01-03
5,31-AUG-2024,2024-08-31
6,30-SEP-2024,2024-09-30
8,05-SEP-2024,2024-09-05
10,14-FEB-2025,2025-02-14


 

**The demo should display this output.**  

<div style="overflow-x:auto;">
    <style>
        table {
            border-collapse: collapse;
            margin: 15px 0;
            font-size: 0.9em;
            font-family: sans-serif;
            width: auto;
        }
        th, td {
            padding: 8px;
            text-align: right;
            border-bottom: 1px solid #ddd;
        }
        tr:hover {background-color: #f5f5f5;}
    </style>
    <table border="0" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>post_until</th>
      <th>post_until_converted</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>3</th>
      <td>03-JAN-2025</td>
      <td>2025-01-03</td>
    </tr>
    <tr>
      <th>5</th>
      <td>31-AUG-2024</td>
      <td>2024-08-31</td>
    </tr>
    <tr>
      <th>6</th>
      <td>30-SEP-2024</td>
      <td>2024-09-30</td>
    </tr>
    <tr>
      <th>8</th>
      <td>05-SEP-2024</td>
      <td>2024-09-05</td>
    </tr>
    <tr>
      <th>10</th>
      <td>14-FEB-2025</td>
      <td>2025-02-14</td>
    </tr>
  </tbody>
</table>
    </div>


 ---
 <!-- Test Cell Boilerplate -->  
The cell below will test your solution for convert_post_until (exercise 2). The testing variables will be available for debugging under the following names in a dictionary format.  
- `input_vars` - Input variables for your solution.   
- `original_input_vars` - Copy of input variables from prior to running your solution. Any `key:value` pair in `original_input_vars` should also exist in `input_vars` - otherwise the inputs were modified by your solution.  
- `returned_output_vars` - Outputs returned by your solution.  
- `true_output_vars` - The expected output. This _should_ "match" `returned_output_vars` based on the question requirements - otherwise, your solution is not returning the correct output. 


In [12]:
### Test Cell - Exercise 2  


from cse6040_devkit.tester_fw.testers import Tester
from yaml import safe_load
from time import time

tracemalloc.start()
mem_start, peak_start = tracemalloc.get_traced_memory()
print(f"initial memory usage: {mem_start/1024/1024:.2f} MB")

# Load testing utility
with open('resource/asnlib/publicdata/execute_tests', 'rb') as f:
    executor = dill.load(f)

@run_with_timeout(error_threshold=200.0, warning_threshold=100.0)
@suppress_stdout
def execute_tests(**kwargs):
    return executor(**kwargs)


# Execute test
start_time = time()
passed, test_case_vars, e = execute_tests(func=plugins.sql_executor(convert_post_until_query),
              ex_name='convert_post_until',
              key=b'btJ0h55N40azEpSYp7KWZGFYuc4IQesVgtsBQFmcS-8=', 
              n_iter=10)
# Assign test case vars for debugging
input_vars, original_input_vars, returned_output_vars, true_output_vars = test_case_vars
duration = time() - start_time
print(f"Test duration: {duration:.2f} seconds")
current_memory, peak_memory = tracemalloc.get_traced_memory()
print(f"memory after test: {current_memory/1024/1024:.2f} MB")
print(f"memory peak during test: {peak_memory/1024/1024:.2f} MB")
tracemalloc.stop()
if e: raise e
assert passed, 'The solution to convert_post_until did not pass the test.'

###
### AUTOGRADER TEST - DO NOT REMOVE
###

print('Passed! Please submit.')

initial memory usage: 0.00 MB
Test duration: 0.24 seconds
memory after test: 0.13 MB
memory peak during test: 6.38 MB
Passed! Please submit.


## Handling NULL Values

There are two common situations in SQL where you are likely to encounter NULL values:
1. Some tables have optional fields (which is often a sign that the database is not fully [normalized](https://en.wikipedia.org/wiki/Database_normalization)).
2. You perform a non-inner join, which introduces NULL values.

NULL values affect comparsions and aggregations. It is fundamental to understand why and how to handle these situations.
- Comparsions: any comparsions (e.g. `=`, `!=`, `<`, etc.) with NULL yields NULL. To check for NULL, use `IS NULL`
    ```sql
    SELECT 1 = NULL --result: NULL
    ```
    ```sql
    SELECT NULL = NULL --result: NULL
    ```
    ```sql
    SELECT NULL IS NULL --result: TRUE
    ```
- Aggregations: aggregate functions (e.g. `SUM`, `COUNT`, `AVG`, etc.) usually ignore NULL by default.

In the following exercise, the column `post_until` has some NULL values present (situation 1). We want to determine what percentage of that column has NULLs present in the following exercise. We'll cover situation 2 in more depth later.

### Exercise 3: (1 points)
**post_until_none_pct**  

**Your task:** define `post_until_none_pct_query` as follows:

It should query the `job_postings` table with the following columns in the result:
- `num_post_until_none`: The count of `NULL` values in the `post_until` column from the `job_postings` table. This will be the numerator of your percentage.
- `denom_post_until`: The count of rows in the `job_postings` table. This will be the denominator of your percentage.
- `pct_post_until_null`: The percentage of `NULL` values in the `post_until` column from the `job_postings` table. This should be a decimal rounded to 2 decimal places and it is not an integer.

**Hint**: The easiest way to accomplish this is to use a `CASE` statement within an aggregate function. Here's a hint about [conditional sums](https://www.interviewquery.com/p/sum-case-when-sql).

In [13]:
### Solution - Exercise 3  
post_until_none_pct_query = '''
SELECT SUM(CASE WHEN post_until IS NULL THEN 1 ELSE 0 END) AS num_post_until_none,
COUNT(*) AS denom_post_until,
ROUND(SUM(CASE WHEN post_until IS NULL THEN 1.0 ELSE 0.0 END) / COUNT(*), 2) AS pct_post_until_null
FROM job_postings

'''

### Demo function call
demo_result_post_until_none_pct = pd.read_sql_query(post_until_none_pct_query, conn)
display(demo_result_post_until_none_pct)

,num_post_until_none,denom_post_until,pct_post_until_null
0,3718,5410,0.69


 

**The demo should display this output.**  

<div style="overflow-x:auto;">
    <style>
        table {
            border-collapse: collapse;
            margin: 15px 0;
            font-size: 0.9em;
            font-family: sans-serif;
            width: auto;
        }
        th, td {
            padding: 8px;
            text-align: right;
            border-bottom: 1px solid #ddd;
        }
        tr:hover {background-color: #f5f5f5;}
    </style>
    <table border="0" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>num_post_until_none</th>
      <th>denom_post_until</th>
      <th>pct_post_until_null</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>0</th>
      <td>3718</td>
      <td>5410</td>
      <td>0.69</td>
    </tr>
  </tbody>
</table>
    </div>


 ---
 <!-- Test Cell Boilerplate -->  
The cell below will test your solution for post_until_none_pct (exercise 3). The testing variables will be available for debugging under the following names in a dictionary format.  
- `input_vars` - Input variables for your solution.   
- `original_input_vars` - Copy of input variables from prior to running your solution. Any `key:value` pair in `original_input_vars` should also exist in `input_vars` - otherwise the inputs were modified by your solution.  
- `returned_output_vars` - Outputs returned by your solution.  
- `true_output_vars` - The expected output. This _should_ "match" `returned_output_vars` based on the question requirements - otherwise, your solution is not returning the correct output. 


In [14]:
### Test Cell - Exercise 3  


from cse6040_devkit.tester_fw.testers import Tester
from yaml import safe_load
from time import time

tracemalloc.start()
mem_start, peak_start = tracemalloc.get_traced_memory()
print(f"initial memory usage: {mem_start/1024/1024:.2f} MB")

# Load testing utility
with open('resource/asnlib/publicdata/execute_tests', 'rb') as f:
    executor = dill.load(f)

@run_with_timeout(error_threshold=200.0, warning_threshold=100.0)
@suppress_stdout
def execute_tests(**kwargs):
    return executor(**kwargs)


# Execute test
start_time = time()
passed, test_case_vars, e = execute_tests(func=plugins.sql_executor(post_until_none_pct_query),
              ex_name='post_until_none_pct',
              key=b'btJ0h55N40azEpSYp7KWZGFYuc4IQesVgtsBQFmcS-8=', 
              n_iter=10)
# Assign test case vars for debugging
input_vars, original_input_vars, returned_output_vars, true_output_vars = test_case_vars
duration = time() - start_time
print(f"Test duration: {duration:.2f} seconds")
current_memory, peak_memory = tracemalloc.get_traced_memory()
print(f"memory after test: {current_memory/1024/1024:.2f} MB")
print(f"memory peak during test: {peak_memory/1024/1024:.2f} MB")
tracemalloc.stop()
if e: raise e
assert passed, 'The solution to post_until_none_pct did not pass the test.'

###
### AUTOGRADER TEST - DO NOT REMOVE
###

print('Passed! Please submit.')

initial memory usage: 0.00 MB
Test duration: 0.18 seconds
memory after test: 0.07 MB
memory peak during test: 2.52 MB
Passed! Please submit.


### Introduction to Inline Subqueries and Common Table Expressions (CTEs)

When working with SQL, it's common to encounter situations where you need to perform calculations or data transformations based on grouped or filtered data. Two powerful tools for handling such scenarios are **inline subqueries** and **Common Table Expressions (CTEs)**. Both approaches allow you to structure your queries in a way that breaks down complex operations into manageable steps, but they differ in syntax, reusability, and performance characteristics.

#### Inline Subqueries

An **inline subquery** is a query nested within another SQL query, typically within the `SELECT`, `WHERE`, or `FROM` clause. Inline subqueries allow you to perform calculations or filtering within the context of the outer query. They are particularly useful when you need to filter results based on aggregated values like maximum, minimum, or average.

**Example**: Suppose you have a table `employees` and you want to compare each employee's salary to the highest salary in their department:

```sql
SELECT e.name, 
       e.department, 
       e.salary, 
       dept_max.max_salary
  FROM employees AS e
  JOIN ( SELECT department, 
                MAX(salary) AS max_salary
           FROM employees
          GROUP BY department
       ) AS dept_max 
    ON e.department = dept_max.department
```

This query uses an inline subquery to determine the maximum salary for each department. This intermediate result is joined to the `employees` table to include the maximum department salary for each employee.

Inline subqueries are powerful, but they can become less efficient when the same calculation needs to be repeated multiple times, as each repetition can slow down the query execution. Visit the [SQLite Documentation's Subqueries page](https://www.sqlite.org/lang_expr.html#subquery_expressions) to learn more.

#### Common Table Expressions (CTEs)

A **Common Table Expression (CTE)**, introduced by the `WITH` clause, is a named temporary result set that you can reference within your main query. CTEs improve the readability and maintainability of SQL code, especially for complex queries, by allowing you to break down operations into logical steps.

**Example**: The same result as the inline subquery example above can be achieved using a CTE:

```sql
WITH MaxSalaries AS ( SELECT department, 
                             MAX(salary) AS max_salary
                        FROM employees
                       GROUP BY department
)

SELECT e.name,
       e.department, 
       e.salary, 
       ms.max_salary
  FROM employees AS e
  JOIN MaxSalaries AS ms 
    ON e.department = ms.department
```

This CTE first calculates the maximum salary for each department, then the main query joins this result with the original `employees` table to show each employee along with their department's maximum salary.

CTEs are often preferred for their clarity and efficiency, particularly when the same calculation or transformation needs to be referenced multiple times within a query. Visit the [SQLite Documentation's CTE page](https://www.sqlite.org/lang_with.html) to learn more.

The upcoming exercises will demonstrate these concepts further.

### Exercise 4: (1 points)
**max_salary_by_category_inline**  

**Your task:** define `max_salary_by_category_inline_query` as follows:

It should query the database to create a result derived from the `salaries`, `jobs`, `job_postings`, and `job_titles` tables with the following columns:
- `business_title`: The business title from the `job_titles` table.
- `job_category`: The job category from the `job_titles` table.
- `salary_range_to`: The salary range maximum (`salary_range_to`) for each individual job posting from the `salaries` table.
- `max_salary`: The maximum salary range for all job postings with the same `job_category` from the `salaries` table

**Requirements/steps**:
- JOIN the `salaries`, `jobs`, `job_postings`, and `job_titles` tables to correctly match each job posting with its corresponding title and salary.
- Use an inline subquery to determine the maximum value of `salary_range_to` for each job category.
  - The **subquery** must contain an open parenthesis followed by the SELECT keyword.
  - The query must not contain the WITH keyword.
- The query should use the `DISTINCT` keyword or some other means to ensure that there are no duplicate values.
- ORDER BY `job_category` ASCENDING then `business_title` ASCENDING

**Hint**: The following SQLite functions and concepts will help: [JOIN](https://www.sqlite.org/syntax/join-operator.html), [MAX](https://www.sqlite.org/lang_aggfunc.html#max), and [SUBQUERY](https://www.sqlite.org/lang_expr.html#subqueries).

In [15]:
### Solution - Exercise 4  
max_salary_by_category_inline_query = '''
SELECT DISTINCT
    jt.business_title,
    jt.job_category,
    s.salary_range_to,
    (SELECT MAX(s_sub.salary_range_to)
     FROM salaries s_sub
     JOIN job_postings jp_sub ON s_sub.job_id = jp_sub.job_id
     JOIN jobs j_sub ON jp_sub.job_id = j_sub.job_id
     JOIN job_titles jt_sub ON j_sub.job_title_id = jt_sub.job_title_id
     WHERE jt_sub.job_category = jt.job_category) AS max_salary
FROM salaries s
JOIN job_postings jp ON s.job_id = jp.job_id
JOIN jobs j ON jp.job_id = j.job_id
JOIN job_titles jt ON j.job_title_id = jt.job_title_id
ORDER BY jt.job_category ASC, jt.business_title ASC;

'''

### Demo function call
demo_result_max_salary_by_category_inline = pd.read_sql_query(max_salary_by_category_inline_query, conn)
display(demo_result_max_salary_by_category_inline.head())

,business_title,job_category,salary_range_to,max_salary
0,"ASSISTANT DEPUTY COMMISSIONER-ADMINISTRATION, ...",Administration & Human Resources,153107.0,271736.0
1,"ASSISTANT DIRECTOR, SALARY ADMINISTRATION",Administration & Human Resources,90363.0,271736.0
2,"ASSOCIATE DIRECTOR OF PAYROLL, TIMEKEEPING & B...",Administration & Human Resources,115000.0,271736.0
3,Administration Services Liaison,Administration & Human Resources,91768.0,271736.0
4,Assistant Civil Service Coordinator,Administration & Human Resources,71840.0,271736.0


 

**The demo should display this output.**  

<div style="overflow-x:auto;">
    <style>
        table {
            border-collapse: collapse;
            margin: 15px 0;
            font-size: 0.9em;
            font-family: sans-serif;
            width: auto;
        }
        th, td {
            padding: 8px;
            text-align: right;
            border-bottom: 1px solid #ddd;
        }
        tr:hover {background-color: #f5f5f5;}
    </style>
    <table border="0" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>business_title</th>
      <th>job_category</th>
      <th>salary_range_to</th>
      <th>max_salary</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>0</th>
      <td>ASSISTANT DEPUTY COMMISSIONER-ADMINISTRATION, PLANNING &amp; OPERATIONS</td>
      <td>Administration &amp; Human Resources</td>
      <td>153107.00</td>
      <td>271736.00</td>
    </tr>
    <tr>
      <th>1</th>
      <td>ASSISTANT DIRECTOR, SALARY ADMINISTRATION</td>
      <td>Administration &amp; Human Resources</td>
      <td>90363.00</td>
      <td>271736.00</td>
    </tr>
    <tr>
      <th>2</th>
      <td>ASSOCIATE DIRECTOR OF PAYROLL, TIMEKEEPING &amp; BENEFITS</td>
      <td>Administration &amp; Human Resources</td>
      <td>115000.00</td>
      <td>271736.00</td>
    </tr>
    <tr>
      <th>3</th>
      <td>Administration Services Liaison</td>
      <td>Administration &amp; Human Resources</td>
      <td>91768.00</td>
      <td>271736.00</td>
    </tr>
    <tr>
      <th>4</th>
      <td>Assistant Civil Service Coordinator</td>
      <td>Administration &amp; Human Resources</td>
      <td>71840.00</td>
      <td>271736.00</td>
    </tr>
  </tbody>
</table>
    </div>


 ---
 <!-- Test Cell Boilerplate -->  
The cell below will test your solution for max_salary_by_category_inline (exercise 4). The testing variables will be available for debugging under the following names in a dictionary format.  
- `input_vars` - Input variables for your solution.   
- `original_input_vars` - Copy of input variables from prior to running your solution. Any `key:value` pair in `original_input_vars` should also exist in `input_vars` - otherwise the inputs were modified by your solution.  
- `returned_output_vars` - Outputs returned by your solution.  
- `true_output_vars` - The expected output. This _should_ "match" `returned_output_vars` based on the question requirements - otherwise, your solution is not returning the correct output. 


In [16]:
### Test Cell - Exercise 4  


from cse6040_devkit.tester_fw.testers import Tester
from yaml import safe_load
from time import time

tracemalloc.start()
mem_start, peak_start = tracemalloc.get_traced_memory()
print(f"initial memory usage: {mem_start/1024/1024:.2f} MB")

# Load testing utility
with open('resource/asnlib/publicdata/execute_tests', 'rb') as f:
    executor = dill.load(f)

@run_with_timeout(error_threshold=200.0, warning_threshold=100.0)
@suppress_stdout
def execute_tests(**kwargs):
    return executor(**kwargs)


plugin_kwargs = utils.load_object_from_publicdata('max_salary_by_category_inline_plugin_kwargs')

# Execute test
start_time = time()
passed, test_case_vars, e = execute_tests(func=plugins.sql_validator(max_salary_by_category_inline_query, **plugin_kwargs),
              ex_name='max_salary_by_category_inline',
              key=b'btJ0h55N40azEpSYp7KWZGFYuc4IQesVgtsBQFmcS-8=', 
              n_iter=10)
# Assign test case vars for debugging
input_vars, original_input_vars, returned_output_vars, true_output_vars = test_case_vars
duration = time() - start_time
print(f"Test duration: {duration:.2f} seconds")
current_memory, peak_memory = tracemalloc.get_traced_memory()
print(f"memory after test: {current_memory/1024/1024:.2f} MB")
print(f"memory peak during test: {peak_memory/1024/1024:.2f} MB")
tracemalloc.stop()
if e: raise e
assert passed, 'The solution to max_salary_by_category_inline did not pass the test.'

###
### AUTOGRADER TEST - DO NOT REMOVE
###

print('Passed! Please submit.')

initial memory usage: 0.00 MB
Test duration: 0.78 seconds
memory after test: 0.29 MB
memory peak during test: 75.84 MB
Passed! Please submit.


### Exercise 5: (1 points)
**max_salary_by_category_cte**  

**Your task:** define `max_salary_by_category_cte_query` as follows:

It should query the database to create a new DataFrame from the `salaries`, `jobs`, `job_postings`, and `job_titles` tables with the following columns:
- `business_title`: The business_title from the `job_titles` table.
- `job_category`: The job category from the `job_titles` table.
- `salary_range_to`: The salary range maximum (`salary_range_to`) for each individual job posting from the `salaries` table.
- `max_salary`: The maximum salary range for all job postings with the same `job_category` from the `salaries` tables as calculated by your CTE.


**Requirements/steps**:
- JOIN the `salaries`, `jobs`, `job_postings`, and `job_titles` tables to correctly match each job posting with its corresponding title and salary.
- Use a CTE to determine the maximum value of `salary_range_to` for each job category.
- The query must contain the `WITH` keyword.
- The query should use the `DISTINCT` keyword to ensure that there are no duplicate values.
- ORDER BY `job_category` ASCENDING then `business_title` ASCENDING

**Hint**: The following SQLite functions and concepts will help: [JOIN](https://www.sqlite.org/syntax/join-operator.html), [MAX](https://www.sqlite.org/lang_aggfunc.html#max), and [CTE/WITH](https://sqlite.org/lang_with.html).

In [17]:
### Solution - Exercise 5  
max_salary_by_category_cte_query = '''
WITH max_salary_df AS (SELECT jt_sub.job_category,
     MAX(s_sub.salary_range_to) AS max_salary
     FROM salaries s_sub
     JOIN job_postings AS jp_sub ON s_sub.job_id = jp_sub.job_id
     JOIN jobs AS j_sub ON jp_sub.job_id = j_sub.job_id
     JOIN job_titles AS jt_sub ON j_sub.job_title_id = jt_sub.job_title_id
     GROUP BY jt_sub.job_category)

SELECT DISTINCT
    jt.business_title,
    jt.job_category,
    s.salary_range_to,
    ms.max_salary
FROM salaries s
JOIN job_postings AS jp ON s.job_id = jp.job_id
JOIN jobs AS j ON jp.job_id = j.job_id
JOIN job_titles AS jt ON j.job_title_id = jt.job_title_id
JOIN max_salary_df AS ms ON ms.job_category = jt.job_category
ORDER BY jt.job_category ASC, jt.business_title ASC;

'''

### Demo function call
demo_result_max_salary_by_category_cte = pd.read_sql_query(max_salary_by_category_cte_query, conn)
display(demo_result_max_salary_by_category_cte.head())

,business_title,job_category,salary_range_to,max_salary
0,"ASSISTANT DEPUTY COMMISSIONER-ADMINISTRATION, ...",Administration & Human Resources,153107.0,271736.0
1,"ASSISTANT DIRECTOR, SALARY ADMINISTRATION",Administration & Human Resources,90363.0,271736.0
2,"ASSOCIATE DIRECTOR OF PAYROLL, TIMEKEEPING & B...",Administration & Human Resources,115000.0,271736.0
3,Administration Services Liaison,Administration & Human Resources,91768.0,271736.0
4,Assistant Civil Service Coordinator,Administration & Human Resources,71840.0,271736.0


 

**The demo should display this output.**  

<div style="overflow-x:auto;">
    <style>
        table {
            border-collapse: collapse;
            margin: 15px 0;
            font-size: 0.9em;
            font-family: sans-serif;
            width: auto;
        }
        th, td {
            padding: 8px;
            text-align: right;
            border-bottom: 1px solid #ddd;
        }
        tr:hover {background-color: #f5f5f5;}
    </style>
    <table border="0" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>business_title</th>
      <th>job_category</th>
      <th>salary_range_to</th>
      <th>max_salary</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>0</th>
      <td>ASSISTANT DEPUTY COMMISSIONER-ADMINISTRATION, PLANNING &amp; OPERATIONS</td>
      <td>Administration &amp; Human Resources</td>
      <td>153107.00</td>
      <td>271736.00</td>
    </tr>
    <tr>
      <th>1</th>
      <td>ASSISTANT DIRECTOR, SALARY ADMINISTRATION</td>
      <td>Administration &amp; Human Resources</td>
      <td>90363.00</td>
      <td>271736.00</td>
    </tr>
    <tr>
      <th>2</th>
      <td>ASSOCIATE DIRECTOR OF PAYROLL, TIMEKEEPING &amp; BENEFITS</td>
      <td>Administration &amp; Human Resources</td>
      <td>115000.00</td>
      <td>271736.00</td>
    </tr>
    <tr>
      <th>3</th>
      <td>Administration Services Liaison</td>
      <td>Administration &amp; Human Resources</td>
      <td>91768.00</td>
      <td>271736.00</td>
    </tr>
    <tr>
      <th>4</th>
      <td>Assistant Civil Service Coordinator</td>
      <td>Administration &amp; Human Resources</td>
      <td>71840.00</td>
      <td>271736.00</td>
    </tr>
  </tbody>
</table>
    </div>


 ---
 <!-- Test Cell Boilerplate -->  
The cell below will test your solution for max_salary_by_category_cte (exercise 5). The testing variables will be available for debugging under the following names in a dictionary format.  
- `input_vars` - Input variables for your solution.   
- `original_input_vars` - Copy of input variables from prior to running your solution. Any `key:value` pair in `original_input_vars` should also exist in `input_vars` - otherwise the inputs were modified by your solution.  
- `returned_output_vars` - Outputs returned by your solution.  
- `true_output_vars` - The expected output. This _should_ "match" `returned_output_vars` based on the question requirements - otherwise, your solution is not returning the correct output. 


In [18]:
### Test Cell - Exercise 5  


from cse6040_devkit.tester_fw.testers import Tester
from yaml import safe_load
from time import time

tracemalloc.start()
mem_start, peak_start = tracemalloc.get_traced_memory()
print(f"initial memory usage: {mem_start/1024/1024:.2f} MB")

# Load testing utility
with open('resource/asnlib/publicdata/execute_tests', 'rb') as f:
    executor = dill.load(f)

@run_with_timeout(error_threshold=200.0, warning_threshold=100.0)
@suppress_stdout
def execute_tests(**kwargs):
    return executor(**kwargs)


plugin_kwargs = utils.load_object_from_publicdata('max_salary_by_category_cte_plugin_kwargs')

# Execute test
start_time = time()
passed, test_case_vars, e = execute_tests(func=plugins.sql_validator(max_salary_by_category_cte_query, **plugin_kwargs),
              ex_name='max_salary_by_category_cte',
              key=b'btJ0h55N40azEpSYp7KWZGFYuc4IQesVgtsBQFmcS-8=', 
              n_iter=10)
# Assign test case vars for debugging
input_vars, original_input_vars, returned_output_vars, true_output_vars = test_case_vars
duration = time() - start_time
print(f"Test duration: {duration:.2f} seconds")
current_memory, peak_memory = tracemalloc.get_traced_memory()
print(f"memory after test: {current_memory/1024/1024:.2f} MB")
print(f"memory peak during test: {peak_memory/1024/1024:.2f} MB")
tracemalloc.stop()
if e: raise e
assert passed, 'The solution to max_salary_by_category_cte did not pass the test.'

###
### AUTOGRADER TEST - DO NOT REMOVE
###

print('Passed! Please submit.')

initial memory usage: 0.00 MB
Test duration: 0.66 seconds
memory after test: 0.17 MB
memory peak during test: 75.83 MB
Passed! Please submit.


### Introduction to Window Functions

Window functions in SQLite provide a way to perform calculations across a set of table rows that are related to the current row. Unlike aggregate functions, which return a single result for a group of rows, window functions return a result for each individual row while still having access to information about related rows, providing more flexibility in your analysis. Common use cases for window functions include calculating running totals, ranking, and moving averages.

Window functions are defined using the `OVER` clause, which specifies the partitioning and ordering of rows within the window. The most commonly used window functions in SQLite include `ROW_NUMBER()`, `RANK()`, and `SUM()`. These functions can be combined with `PARTITION BY` to break the data into subsets and `ORDER BY` to define the sequence in which rows are processed.

#### Example of a Window Function

Suppose you have a table called `sales` that contains the columns `employee_id`, `sale_date`, and `amount`. You want to calculate the cumulative sales for each employee over time. Here’s a simple example using the `SUM()` window function:

```sql
SELECT employee_id,
       sale_date,
       amount,
       SUM(amount) OVER (
                          PARTITION BY employee_id 
                              ORDER BY sale_date
                        ) AS cumulative_sales
  FROM sales
 ORDER BY employee_id, 
          sale_date
```

In this example:
- `SUM(amount)` calculates the running total of `amount`.
- The `OVER` clause specifies that the window function should be applied to each `employee_id` individually (`PARTITION BY employee_id`) and should consider the order of `sale_date` (`ORDER BY sale_date`).
- The result is a new column, `cumulative_sales`, that shows the cumulative sales amount for each employee over time.

Vist the [SQLite Window Functions Page](https://www.sqlite.org/windowfunctions.html) to learn more.

The following exercise will explore this concept further.

### Exercise 6: (1 points)
**ranks_by_classification**  

**Your task:** define `ranks_by_classification_query` as follows:

It should query the database to create a result derived from the `salaries`, `jobs`, `job_postings`, and `job_titles` tables with the following columns:
- `job_id`: The job_id from the `job_postings` table.
- `title_classification`: The classification for each title in the `job_titles` table.
- `business_title`: The business title for each title in the `job_titles` table.
- `posting_updated`: The value for `posting_updated` in the `job_postings` table.
- `maximum_salary`: The value for `salary_range_to` in the `salaries` table.
- `job_rank`: The ranking for each position within its title classification, based on the criteria specified in the requirements.

**Requirements/steps**:
- The query should join the `job_postings`, `jobs`, `job_titles`, and `salaries` tables to correctly match the job title ids and their corresponding salaries.
- You will need to use the `RANK()` window function to determine the rankings for the jobs.
    - You should `PARTITION BY` the job posting's title classification.
    - You should order the partitions by:
        - The maximum salary column (descending)
        - The `posting_updated` column (descending)
        - The job ID (ascending)
- The query should use the `DISTINCT` keyword or some other means to ensure that there are no duplicate values.
- ORDER BY `job_id` ASCENDING then `job_rank` ASCENDING

**Hint**: You will need to use the [OVER](https://sqlite.org/windowfunctions.html) keyword to properly define the windows.

In [19]:
### Solution - Exercise 6  
ranks_by_classification_query = '''
SELECT DISTINCT
    j.job_id,
    jt.title_classification,
    jt.business_title, 
    jp.posting_updated,
    s.salary_range_to AS maximum_salary,
RANK() OVER (
    PARTITION BY jt.title_classification
    ORDER BY s.salary_range_to DESC, jp.posting_updated DESC, j.job_id ASC
) AS job_rank
FROM job_postings AS jp
    JOIN jobs AS j ON j.job_id = jp.job_id
    JOIN job_titles AS jt ON jt.job_title_id = j.job_title_id
    JOIN salaries AS s ON s.job_id = j.job_id
ORDER BY j.job_id ASC, job_rank ASC;

'''


### Demo function call
demo_result_ranks_by_classification = pd.read_sql_query(ranks_by_classification_query, conn)
display(demo_result_ranks_by_classification.head())

,job_id,title_classification,business_title,posting_updated,maximum_salary,job_rank
0,469953,Competitive-1,CONTRACT ANALYST,2024-08-21 00:00:00.000000,71556.0,2325
1,481622,Non-Competitive-5,Child Protective Manager,2022-06-30 00:00:00.000000,102226.0,503
2,483894,Competitive-1,SENIOR PROJECT MANAGER,2023-12-14 00:00:00.000000,119610.0,713
3,484513,Competitive-1,CONTROL CLERK,2022-09-21 00:00:00.000000,41848.0,3279
4,487203,Competitive-1,UNIT CLERK,2022-07-15 00:00:00.000000,41848.0,3283


 

**The demo should display this output.**  

<div style="overflow-x:auto;">
    <style>
        table {
            border-collapse: collapse;
            margin: 15px 0;
            font-size: 0.9em;
            font-family: sans-serif;
            width: auto;
        }
        th, td {
            padding: 8px;
            text-align: right;
            border-bottom: 1px solid #ddd;
        }
        tr:hover {background-color: #f5f5f5;}
    </style>
    <table border="0" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>job_id</th>
      <th>title_classification</th>
      <th>business_title</th>
      <th>posting_updated</th>
      <th>maximum_salary</th>
      <th>job_rank</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>0</th>
      <td>469953</td>
      <td>Competitive-1</td>
      <td>CONTRACT ANALYST</td>
      <td>2024-08-21 00:00:00.000000</td>
      <td>71556.00</td>
      <td>2325</td>
    </tr>
    <tr>
      <th>1</th>
      <td>481622</td>
      <td>Non-Competitive-5</td>
      <td>Child Protective Manager</td>
      <td>2022-06-30 00:00:00.000000</td>
      <td>102226.00</td>
      <td>503</td>
    </tr>
    <tr>
      <th>2</th>
      <td>483894</td>
      <td>Competitive-1</td>
      <td>SENIOR PROJECT MANAGER</td>
      <td>2023-12-14 00:00:00.000000</td>
      <td>119610.00</td>
      <td>713</td>
    </tr>
    <tr>
      <th>3</th>
      <td>484513</td>
      <td>Competitive-1</td>
      <td>CONTROL CLERK</td>
      <td>2022-09-21 00:00:00.000000</td>
      <td>41848.00</td>
      <td>3279</td>
    </tr>
    <tr>
      <th>4</th>
      <td>487203</td>
      <td>Competitive-1</td>
      <td>UNIT CLERK</td>
      <td>2022-07-15 00:00:00.000000</td>
      <td>41848.00</td>
      <td>3283</td>
    </tr>
  </tbody>
</table>
    </div>


 ---
 <!-- Test Cell Boilerplate -->  
The cell below will test your solution for ranks_by_classification (exercise 6). The testing variables will be available for debugging under the following names in a dictionary format.  
- `input_vars` - Input variables for your solution.   
- `original_input_vars` - Copy of input variables from prior to running your solution. Any `key:value` pair in `original_input_vars` should also exist in `input_vars` - otherwise the inputs were modified by your solution.  
- `returned_output_vars` - Outputs returned by your solution.  
- `true_output_vars` - The expected output. This _should_ "match" `returned_output_vars` based on the question requirements - otherwise, your solution is not returning the correct output. 


In [20]:
### Test Cell - Exercise 6  


from cse6040_devkit.tester_fw.testers import Tester
from yaml import safe_load
from time import time

tracemalloc.start()
mem_start, peak_start = tracemalloc.get_traced_memory()
print(f"initial memory usage: {mem_start/1024/1024:.2f} MB")

# Load testing utility
with open('resource/asnlib/publicdata/execute_tests', 'rb') as f:
    executor = dill.load(f)

@run_with_timeout(error_threshold=200.0, warning_threshold=100.0)
@suppress_stdout
def execute_tests(**kwargs):
    return executor(**kwargs)


# Execute test
start_time = time()
passed, test_case_vars, e = execute_tests(func=plugins.sql_executor(ranks_by_classification_query),
              ex_name='ranks_by_classification',
              key=b'btJ0h55N40azEpSYp7KWZGFYuc4IQesVgtsBQFmcS-8=', 
              n_iter=10)
# Assign test case vars for debugging
input_vars, original_input_vars, returned_output_vars, true_output_vars = test_case_vars
duration = time() - start_time
print(f"Test duration: {duration:.2f} seconds")
current_memory, peak_memory = tracemalloc.get_traced_memory()
print(f"memory after test: {current_memory/1024/1024:.2f} MB")
print(f"memory peak during test: {peak_memory/1024/1024:.2f} MB")
tracemalloc.stop()
if e: raise e
assert passed, 'The solution to ranks_by_classification did not pass the test.'

###
### AUTOGRADER TEST - DO NOT REMOVE
###

print('Passed! Please submit.')

initial memory usage: 0.00 MB
Test duration: 0.75 seconds
memory after test: 0.23 MB
memory peak during test: 76.00 MB
Passed! Please submit.



### Handling NULL Values continued

As previously mentioned, there are two common situations in SQL where you are likely to encounter NULL values:
- Some tables have an optional field (which is often a sign that the database is not fully [normalized](https://en.wikipedia.org/wiki/Database_normalization)).
- You perform a non-inner join, which introduces NULL.

NULL values can cause problems when you attempt to perform certain operations. For example, arithmetic operations with NULL values result in NULL.

There are two main tools which SQLite provides for handling default values for NULLs: [`IFNULL(X, Y)`](https://www.sqlite.org/lang_corefunc.html#ifnull) and [`COALESCE(X, Y, ...)`](https://www.sqlite.org/lang_corefunc.html#coalesce). Both make it possible to provide default values for NULL instances. `COALESCE()` is especially powerful, as it will return the *first non-NULL instance* it encounters in its arguments. This makes it possible to provide multiple fall-back options for NULL values.

> Recall that we have already spent some time discussing how to deal with missing values in native Python data structures, such as by using [default dictionaries](https://docs.python.org/3/library/collections.html#collections.defaultdict) and methods like [`dict.get()`](https://docs.python.org/3/library/stdtypes.html#dict.get). Pandas has a [full section of documentation](https://pandas.pydata.org/pandas-docs/stable/user_guide/missing_data.html) dedicated to handling missing values.

> You can see how SQLite handles operations with NULL values, alongside how other databases handle the same behavior, in their [documentation](https://www.sqlite.org/nulls.html).

> `IFNULL(X, Y)` is behaves nearly functionally identical to `COALESCE(X, Y)`. However, `COALESCE()` can accept an arbitrary number of arguments greater than or equal to 2; `IFNULL()`, by contrast, requires precisely 2.

Consider the following example: we want to know what skills are required for each job posting contained in the `job_postings` table. We have two columns in the table which detail job requirements:

1. `preferred_skills`, which gives us a precise description of what skills a desirable candidate will possess.
2. `minimum_qual_requirements`, which provides a more general description of what is required for the position.

The following query will attempt to display the preferred skills wherever possible. If they are not available, it will fall back to the minimum requirements. If neither are available, it will tell us that the requirements are unknown.

```sql
SELECT job_id,
       COALESCE( preferred_skills,
                 minimum_qual_requirements,
                 'UNKNOWN!'
               ) AS job_requirements
  FROM job_posting
```

The following exercise will demonstrate this concept further.

### Exercise 7: (1 points)
**job_location_details**  

**Your task:** define `job_location_details_query` as follows:

It should query the database to create a result derived from the `job_postings`, `jobs`, and `locations` tables with the following columns:
- `job_id`: The id from the `job_postings` table.
- `location`: The location of the posting, with `NULL` values replaced using a fallback hierarchy. It should be equal to:
    - `work_location_1` in the `jobs` table, if available.
    - Otherwise, `work_location` in the `locations` table, if available.
    - Otherwise, set the value to the string, `UNKNOWN`.

**Requirements/steps**:
- The query should use a `JOIN` between `job_postings` and `jobs` tables and then `LEFT JOIN` with the `locations` table to properly match the locations with the postings.
- You can use either `IFNULL` or `COALESCE` to solve this problem.
- ORDER BY `job_id` ASCENDING then `location` ASCENDING

In [21]:
### Solution - Exercise 7  
job_location_details_query = '''
SELECT jp.job_id,
COALESCE (j.work_location_1,
          l.work_location,
        'UNKNOWN') AS location
FROM job_postings AS jp
JOIN jobs AS j ON j.job_id = jp.job_id
LEFT JOIN locations AS l ON j.location_id = l.location_id
ORDER BY jp.job_id ASC, location ASC 

'''

### Demo function call
demo_result_job_location_details = pd.read_sql_query(job_location_details_query, conn)
display(demo_result_job_location_details.head(10))

,job_id,location
0,469953,4 World Trade Center
1,469953,4 World Trade Center
2,481622,2501 Grand Concourse
3,481622,2501 Grand Concourse
4,483894,4 World Trade Center
5,483894,4 World Trade Center
6,484513,"505 Clermont Avenue, Brooklyn, NY, 11238"
7,484513,"505 Clermont Avenue, Brooklyn, NY, 11238"
8,487203,"505 Clermont Avenue, Brooklyn, NY, 11238"
9,487203,"505 Clermont Avenue, Brooklyn, NY, 11238"


 

**The demo should display this output.**  

<div style="overflow-x:auto;">
    <style>
        table {
            border-collapse: collapse;
            margin: 15px 0;
            font-size: 0.9em;
            font-family: sans-serif;
            width: auto;
        }
        th, td {
            padding: 8px;
            text-align: right;
            border-bottom: 1px solid #ddd;
        }
        tr:hover {background-color: #f5f5f5;}
    </style>
    <table border="0" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>job_id</th>
      <th>location</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>0</th>
      <td>469953</td>
      <td>4 World Trade Center</td>
    </tr>
    <tr>
      <th>1</th>
      <td>469953</td>
      <td>4 World Trade Center</td>
    </tr>
    <tr>
      <th>2</th>
      <td>481622</td>
      <td>2501 Grand Concourse</td>
    </tr>
    <tr>
      <th>3</th>
      <td>481622</td>
      <td>2501 Grand Concourse</td>
    </tr>
    <tr>
      <th>4</th>
      <td>483894</td>
      <td>4 World Trade Center</td>
    </tr>
    <tr>
      <th>5</th>
      <td>483894</td>
      <td>4 World Trade Center</td>
    </tr>
    <tr>
      <th>6</th>
      <td>484513</td>
      <td>505 Clermont Avenue, Brooklyn, NY, 11238</td>
    </tr>
    <tr>
      <th>7</th>
      <td>484513</td>
      <td>505 Clermont Avenue, Brooklyn, NY, 11238</td>
    </tr>
    <tr>
      <th>8</th>
      <td>487203</td>
      <td>505 Clermont Avenue, Brooklyn, NY, 11238</td>
    </tr>
    <tr>
      <th>9</th>
      <td>487203</td>
      <td>505 Clermont Avenue, Brooklyn, NY, 11238</td>
    </tr>
  </tbody>
</table>
    </div>


 ---
 <!-- Test Cell Boilerplate -->  
The cell below will test your solution for job_location_details (exercise 7). The testing variables will be available for debugging under the following names in a dictionary format.  
- `input_vars` - Input variables for your solution.   
- `original_input_vars` - Copy of input variables from prior to running your solution. Any `key:value` pair in `original_input_vars` should also exist in `input_vars` - otherwise the inputs were modified by your solution.  
- `returned_output_vars` - Outputs returned by your solution.  
- `true_output_vars` - The expected output. This _should_ "match" `returned_output_vars` based on the question requirements - otherwise, your solution is not returning the correct output. 


In [22]:
### Test Cell - Exercise 7  


from cse6040_devkit.tester_fw.testers import Tester
from yaml import safe_load
from time import time

tracemalloc.start()
mem_start, peak_start = tracemalloc.get_traced_memory()
print(f"initial memory usage: {mem_start/1024/1024:.2f} MB")

# Load testing utility
with open('resource/asnlib/publicdata/execute_tests', 'rb') as f:
    executor = dill.load(f)

@run_with_timeout(error_threshold=200.0, warning_threshold=100.0)
@suppress_stdout
def execute_tests(**kwargs):
    return executor(**kwargs)


# Execute test
start_time = time()
passed, test_case_vars, e = execute_tests(func=plugins.sql_executor(job_location_details_query),
              ex_name='job_location_details',
              key=b'btJ0h55N40azEpSYp7KWZGFYuc4IQesVgtsBQFmcS-8=', 
              n_iter=10)
# Assign test case vars for debugging
input_vars, original_input_vars, returned_output_vars, true_output_vars = test_case_vars
duration = time() - start_time
print(f"Test duration: {duration:.2f} seconds")
current_memory, peak_memory = tracemalloc.get_traced_memory()
print(f"memory after test: {current_memory/1024/1024:.2f} MB")
print(f"memory peak during test: {peak_memory/1024/1024:.2f} MB")
tracemalloc.stop()
if e: raise e
assert passed, 'The solution to job_location_details did not pass the test.'

###
### AUTOGRADER TEST - DO NOT REMOVE
###

print('Passed! Please submit.')

initial memory usage: 0.00 MB
Test duration: 0.96 seconds
memory after test: 0.63 MB
memory peak during test: 106.47 MB
Passed! Please submit.


## Query Logic and Refactoring

### Writing queries for better performance

[Inspiration](https://medium.com/learning-sql/12-tips-for-optimizing-sql-queries-for-faster-performance-8c6c092d7af1) for some of these performance enhancements.

1. Use Indexes: Create [indexes](https://www.sqlitetutorial.net/sqlite-index/) on the columns that are frequently used in `WHERE`, `JOIN`, `ORDER BY`, and `GROUP BY` clauses. 
    ```sql
    -- recall that we earlier showed how to create a SQLite table students with 4 students
    -- c.execute("CREATE TABLE students (gtid INTEGER, student_name TEXT)")
    -- c.execute("INSERT INTO students VALUES (123, 'Vuduc')")
    -- c.execute("INSERT INTO students VALUES (456, 'Chau')")
    -- c.execute("INSERT INTO students VALUES (381, 'Bader')")
    -- c.execute("INSERT INTO students VALUES (991, 'Sokol')")

    -- query that table for information
    SELECT * 
      FROM students 
     WHERE gtid=991
    ```

    While this table is small with only 4 records, if we assume that the table is much larger, this `SELECT` query would have to search the entire table to match the `gtid`. You can create an index on `gtid` to improve this query.

    ```sql
    CREATE INDEX idx_students_gtid
        ON students(gtid);
    ```
    This creates an index on the `gtid` column in the `students` table which speeds up our original `SELECT` statement.

2. Avoid `SELECT *`: Only select the columns that you need in your analysis. Using `SELECT *` can slow down query performance because it returns all columns in a table, including those not needed in the query. For example, if you just need the student's `student_name`, using `SELECT *` on a larger table could negatively affect performance.
    ```sql
    -- this will be slower on a larger table
    SELECT * 
      FROM students 
     WHERE gtid=991

    -- this will be faster on a larger table
    SELECT student_name 
      FROM students 
     WHERE gtid=991 
    ```

3. Filter early with `WHERE`: Reducing the number of rows as early as possible to improve performance in your query. For larger tables where you are looking for students with `gtid=991`, filtering that early will positively affect performance.
    ```sql
    -- this will be slower on a larger table and then filtering in Pandas for gtid=991
    SELECT * 
      FROM students

    -- this will be faster on a larger table
    SELECT student_name 
      FROM students 
     WHERE gtid=991 
    ```

4. Use `EXISTS` or `JOIN` instead of `IN`: Using the `IN` operator can slow down query performance because it requires a full table scan in the subquery. To optimize, you can use the `EXISTS` or `JOIN` operators instead of `IN`.
    ```sql
    -- lets create an honors table
    -- c.execute("CREATE TABLE honors (gtid INTEGER)")
    -- c.execute("INSERT INTO honors VALUES (123)")
    -- c.execute("INSERT INTO honors VALUES (991)")

    -- this query will be slowest
    SELECT * 
      FROM students 
     WHERE gtid IN ( SELECT gtid 
                       FROM honors
                   )

    -- this query will be faster
    SELECT * 
      FROM students 
     WHERE EXISTS ( SELECT 1 
                      FROM honors 
                     WHERE honors.gtid=students.gtid
                  )

    -- this query will be the fastest as it avoids subqueries
    SELECT students.* 
      FROM students 
      JOIN honors 
        ON students.gtid=honors.gtid
    ```
    The `EXISTS` checks to see if a matching row exists in the `honors` table instead of using the `IN` operators. This can enhance query performance by avoiding a full table scan. Using the `JOIN` is more index-friendly and will be faster as it avoids the use of a nested subquery.

5. Use `LIMIT` if possible: Using `LIMIT` prevents you from fetching too many unnecessary rows. This will result in less data to process and return as a result. Using LIMIT was covered previously but the syntax would look like:
    ```sql
    SELECT * 
      FROM students 
     LIMIT 1
    ```

6. Avoid using wildcards if possible: Using wildcards e.g. `%`, `_`; negatively impact performance. This is because SQL has to do a full table scan, thereby resulting in less optimized queries.
    ```sql
    -- this will be the slowest and will return 'Vuduc'
    SELECT * 
      FROM students 
     WHERE student_name LIKE '%V%'

    -- this will be faster and will return 'Vuduc'
    SELECT * 
      FROM students 
     WHERE student_name LIKE 'V%'

    -- this will be the fastest and will return 'Vuduc'
    SELECT * 
      FROM students 
     WHERE student_name >='V' AND student_name <'W'

7. Use `EXPLAIN QUERY PLAN`: Using `EXPLAIN QUERY PLAN your_sql_query` will allow you to understand how SQLite is planning to execute your query. Use that to your advantage.
    ```sql
    -- this will show the steps that SQLite plans on taking to execute the SELECT statement
    EXPLAIN QUERY PLAN 
    SELECT * 
      FROM students 
     WHERE gtid IN ( SELECT gtid 
                       FROM honors
                   )
    ``` 


### EXCEPT vs NOT IN vs LEFT JOIN
What if you need to find out whether something isn't in a table. In our example, let's say we wanted to find students who were not honors students.

1. `EXCEPT`: Returns rows in the first query that are not in the second. It treats NULL as equal to NULL (specifically [*"compound SELECT operators, NULL values are considered equal to other NULL values and distinct from all non-NULL values"*](https://sqlite.org/lang_select.html)) and can slow down on larger sets. This will automatically remove duplicates if any

    ```sql
    SELECT gtid 
      FROM students 
    
    EXCEPT 
    
    SELECT gtid 
      FROM honors
    ```
    This will return the 2 `gtid` for students who were not honors students [381, 456].

2. `NOT IN`: Filters values not present in the list. It doesn't handle NULLs at all and can slow down on larger subqueries. If any NULLs exist in the subquery, **the entire result set will be empty** and no rows will match because comparisons with NULL return UNKNOWN. This will not remove duplicates.

    ```sql
    SELECT gtid 
      FROM students 
     WHERE gtid NOT IN ( SELECT gtid 
                           FROM honors
                       )
    ```
    This will return the 2 `gtid` for students who were not honors students [381, 456].

3. `LEFT JOIN` with `IS NULL`: Returns unmatched rows from a LEFT JOIN. This handles NULLs correctly and is the most efficient especially with configured indexes. This approach does not remove duplicates, but that could be handled with a DISTINCT.

    ```sql
    SELECT students.gtid 
      FROM students 
      LEFT JOIN honors 
        ON students.gtid=honors.gtid 
     WHERE honors.gtid IS NULL
    ```
    This will return the 2 `gtid` for students who were not honors students [381, 456].

In summary, there are a number of trade-offs and considerations when determining the best way to optimize SQLite queries. The main ones aforementioned include: 
1. Data characteristics, including the presence of NULL values
2. Performance requirements
3. Duplicate handling requirements
4. Index availability

## Security Implications and Placeholder Bindings
The well-known [XKCD comic on SQL injection attacks](https://xkcd.com/327/) shown below is a common security concern. In this section, we'll explore the focus of this comic and why the implications are so important.

![xkcd_sql_injection](resource/asnlib/publicdata/xkcd_sql_injection.png)

### Queries and User Input

Imagine that we want to build a Python function which retrieves all of the job titles from our database belonging to a specific career-level. The career-level we are interested in could be any value.

You might realize that we can write a function which takes an argument, `level`, and define the function so that we can specify the career-level *when the function is called*.

> Python has several different mechanisms for interpolating values into strings, such as [f-strings](https://docs.python.org/3/reference/lexical_analysis.html#f-strings) and the [.format() method](https://docs.python.org/3/library/stdtypes.html#str.format). Notice that we also define SQL queries as strings in Python. So, by using string interpolation, we can fill in the blanks of our query with whatever values we want!

Let's try doing this by using string formatting. Then, we will show why this is a bad idea.

The function **get_jobs_by_level_dangerous** is simple enough. The caller gives the desired level and we use a format string to incorporate it directly into the SQL code.  

In [23]:
### Run Me!!!
def get_jobs_by_level_dangerous(level):
    return f"""
SELECT * 
  FROM job_titles 
 WHERE career_level = '{level}'
            """

### Demo function call
entry_level_query = get_jobs_by_level_dangerous('Entry-Level')
pd.read_sql_query(entry_level_query, conn).head()

,job_title_id,business_title,civil_service_title,title_code_no,title_classification,career_level,job_category,level
0,9,CARETAKER X (HA),CARETAKER (HA),90645,Labor-3,Entry-Level,Building Operations & Maintenance,00
1,18,UNIT CLERK,CLERICAL ASSOCIATE,10251,Competitive-1,Entry-Level,Administration & Human Resources Social Services,03
2,27,Asbestos Enforcement Inspector,ASSOCIATE CHEMIST,21822,Competitive-1,Entry-Level,"Policy, Research & Analysis Public Safety, Ins...",01
3,36,CITY PEST CONTROL AIDE,CITY PEST CONTROL AIDE,90643,Labor-3,Entry-Level,Health Building Operations & Maintenance Publi...,00
4,54,HEAVY-DUTY CLEANINGS SERVICES COORDINATOR (HDCSC),COMMUNITY ASSOCIATE,56057,Non-Competitive-5,Entry-Level,Social Services,00


### SQL Injection Attacks
So, why is this a bad idea?

- Consider that we are defining our SQL queries by writing **text**. This text describes *code*, which tells the computer which instructions to execute.
- However, we are describing variables in our query with **text**, too. Then, when we build the query, we are substituting part of our *code* with new *text*.

So... what would happen if the *text* we substituted into our query **was valid SQL code?**

The answer is that our query could potentially do almost *anything*. This is known as a [SQL Injection Attack](https://en.wikipedia.org/wiki/SQL_injection). Using string-formatting with Python to generate SQL queries does not protect against this.

> This is not *necessarily* dangerous. For example, if you are the only person who will ever call this function, then there is no possibility that some user will intentionally try to abuse it. However, if you allow users to interact with this function (such as by collecting their inputs in a web-form), then this approach would let users execute nearly *any* SQL statement against your database.

In this exercise, you will attempt to exploit this vulnerability and obtain more records than you might be expected to access.

### Exercise 8: (1 points)
**malicious_SQL_example**  

**Your task:** define `malicious_SQL_example` as follows:

**The variable `malicious_SQL_example` is a Python string which we will pass as the `level` argument to `get_jobs_by_level_dangerous`. The `level` string should be *purposefully malicious* and take advantage of SQL injection vulnerabilities, as outlined below. This demonstrates how an attacker could use injection to return all records in tables or attempt to modify the database.

**Requirements/steps**:
- You may find the examples in Wikipedia's article on [Incorrectly Structured SQL Statements](https://en.wikipedia.org/wiki/SQL_injection#Technical_implementations) useful (*hint hint!*).
- You should not be returning a full query. Only return the value for `level`, which we will use to build the full query.

In [33]:
### Solution - Exercise 8  
malicious_SQL_example ="' OR '1'='1"

### Demo function call
malicious_query = get_jobs_by_level_dangerous(malicious_SQL_example)
print('Your malicious query:', malicious_query)
demo_result_malicious_SQL_example = pd.read_sql_query(malicious_query, conn)
display(demo_result_malicious_SQL_example.head())

Your malicious query: 
SELECT * 
  FROM job_titles 
 WHERE career_level = '' OR '1'='1'
            


,job_title_id,business_title,civil_service_title,title_code_no,title_classification,career_level,job_category,level
0,1,Senior Design Reviewer (Electrical),ADM ENGINEER (NON MGRL),1001A,Competitive-1,Experienced (non-manager),"Engineering, Architecture, & Planning",00
1,2,Gardener II,GARDENER,81310,Competitive-1,Experienced (non-manager),Constituent Services & Community Programs Gree...,02
2,3,Associate Youth Development Specialist,ASSOCIATE YOUTH DEVELOPMENT SP,52288,Competitive-1,Experienced (non-manager),Social Services,01
3,4,Digital Forensic Examiner,COMMUNITY COORDINATOR,56058,Non-Competitive-5,Experienced (non-manager),"Technology, Data & Innovation Public Safety, I...",00
4,5,Goods & Services Procurement Section Unit Lead,PROCUREMENT ANALYST,12158,Competitive-1,Experienced (non-manager),"Finance, Accounting, & Procurement",03


 

**The demo should display this output.**  

<div style="overflow-x:auto;">
    <style>
        table {
            border-collapse: collapse;
            margin: 15px 0;
            font-size: 0.9em;
            font-family: sans-serif;
            width: auto;
        }
        th, td {
            padding: 8px;
            text-align: right;
            border-bottom: 1px solid #ddd;
        }
        tr:hover {background-color: #f5f5f5;}
    </style>
    <table border="0" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>job_title_id</th>
      <th>business_title</th>
      <th>civil_service_title</th>
      <th>title_code_no</th>
      <th>title_classification</th>
      <th>career_level</th>
      <th>job_category</th>
      <th>level</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>0</th>
      <td>1</td>
      <td>Senior Design Reviewer (Electrical)</td>
      <td>ADM ENGINEER (NON MGRL)</td>
      <td>1001A</td>
      <td>Competitive-1</td>
      <td>Experienced (non-manager)</td>
      <td>Engineering, Architecture, &amp; Planning</td>
      <td>00</td>
    </tr>
    <tr>
      <th>1</th>
      <td>2</td>
      <td>Gardener II</td>
      <td>GARDENER</td>
      <td>81310</td>
      <td>Competitive-1</td>
      <td>Experienced (non-manager)</td>
      <td>Constituent Services &amp; Community Programs Green Jobs Building Operations &amp; Maintenance</td>
      <td>02</td>
    </tr>
    <tr>
      <th>2</th>
      <td>3</td>
      <td>Associate Youth Development Specialist</td>
      <td>ASSOCIATE YOUTH DEVELOPMENT SP</td>
      <td>52288</td>
      <td>Competitive-1</td>
      <td>Experienced (non-manager)</td>
      <td>Social Services</td>
      <td>01</td>
    </tr>
    <tr>
      <th>3</th>
      <td>4</td>
      <td>Digital Forensic Examiner</td>
      <td>COMMUNITY COORDINATOR</td>
      <td>56058</td>
      <td>Non-Competitive-5</td>
      <td>Experienced (non-manager)</td>
      <td>Technology, Data &amp; Innovation Public Safety, Inspections, &amp; Enforcement</td>
      <td>00</td>
    </tr>
    <tr>
      <th>4</th>
      <td>5</td>
      <td>Goods &amp; Services Procurement Section Unit Lead</td>
      <td>PROCUREMENT ANALYST</td>
      <td>12158</td>
      <td>Competitive-1</td>
      <td>Experienced (non-manager)</td>
      <td>Finance, Accounting, &amp; Procurement</td>
      <td>03</td>
    </tr>
  </tbody>
</table>
    </div>

**The demo should display this printed output.**
```
Your malicious query: SELECT * FROM job_titles WHERE career_level = '' OR '1' = '1' --''
```


 ---
 <!-- Test Cell Boilerplate -->  
The cell below will test your solution for malicious_SQL_example (exercise 8). The testing variables will be available for debugging under the following names in a dictionary format.  
- `input_vars` - Input variables for your solution.   
- `original_input_vars` - Copy of input variables from prior to running your solution. Any `key:value` pair in `original_input_vars` should also exist in `input_vars` - otherwise the inputs were modified by your solution.  
- `returned_output_vars` - Outputs returned by your solution.  
- `true_output_vars` - The expected output. This _should_ "match" `returned_output_vars` based on the question requirements - otherwise, your solution is not returning the correct output. 


In [34]:
### Test Cell - Exercise 8  


from cse6040_devkit.tester_fw.testers import Tester
from yaml import safe_load
from time import time

tracemalloc.start()
mem_start, peak_start = tracemalloc.get_traced_memory()
print(f"initial memory usage: {mem_start/1024/1024:.2f} MB")

# Load testing utility
with open('resource/asnlib/publicdata/execute_tests', 'rb') as f:
    executor = dill.load(f)

@run_with_timeout(error_threshold=200.0, warning_threshold=100.0)
@suppress_stdout
def execute_tests(**kwargs):
    return executor(**kwargs)


# Execute test
start_time = time()
passed, test_case_vars, e = execute_tests(func=plugins.malicious_executor(malicious_SQL_example),
              ex_name='malicious_SQL_example',
              key=b'btJ0h55N40azEpSYp7KWZGFYuc4IQesVgtsBQFmcS-8=', 
              n_iter=10)
# Assign test case vars for debugging
input_vars, original_input_vars, returned_output_vars, true_output_vars = test_case_vars
duration = time() - start_time
print(f"Test duration: {duration:.2f} seconds")
current_memory, peak_memory = tracemalloc.get_traced_memory()
print(f"memory after test: {current_memory/1024/1024:.2f} MB")
print(f"memory peak during test: {peak_memory/1024/1024:.2f} MB")
tracemalloc.stop()
if e: raise e
assert passed, 'The solution to malicious_SQL_example did not pass the test.'

###
### AUTOGRADER TEST - DO NOT REMOVE
###

print('Passed! Please submit.')

initial memory usage: 0.00 MB
Test duration: 0.48 seconds
memory after test: 0.34 MB
memory peak during test: 5.28 MB
Passed! Please submit.


### Protecting Against SQL Injections
This is clearly a serious security concern! What can we do about it?

> You should now have enough context to understand why this [XKCD comic](https://xkcd.com/327/) is both funny and insightful; the school-system did not know about injection attacks, and their **entire database was deleted** as a result.

Instead of directly interpolating the values ourselves, we can ask the SQLite API to handle it for us. All major Python-SQL interfaces will provide some mechanism for specifying placeholders in a query. Then, the SQL library will automatically escape special characters and keep your queries from executing unwanted code.

In other words, instead of building the query *ourselves*, we will leave a placeholder in the query and give the SQL API a collection of values to interpolate. We will let the library take care of everything else.

Python's [SQLite3 Module documentation](https://docs.python.org/3/library/sqlite3.html#how-to-use-placeholders-to-bind-values-in-sql-queries) contains examples of how to do this. There are several approaches developers can use, but we will encourage the use of the `?` syntax here.

> The exact method of doing this will vary by SQL backend, so always check the documentation to see how different engines require you to do this. For example, if you are using PostgreSQL, the [Psycopg documentation](https://www.psycopg.org/psycopg3/docs/basic/params.html) will tell you the specific syntax. Tools like [Pandas](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.read_sql.html) will often have a specific function argument for you to use, such as `params`.

### Exercise 9: (1 points)
**get_jobs_by_level_safe**  

**Your task:** define `get_jobs_by_level_safe` as follows:

**Activity**: Query the database connection, selecting every record from the `job_titles` table,
  as long as the `career_level` value is equal to the value specified by `level`. Use placeholders
  for query parameters to prevent SQL injection.

**Inputs**: 
- `level`: A value for `career_level`, as a Python string. We will use it to decide which records to keep.
- `conn`: A `sqlite` connection to a database that has a `job_titles` table.

**Return**: `output`: A tuple with two elements:
- `jobs_by_level_result`: The result of the query operation described above as a Pandas DataFrame.
- `query`: A string containing the query used to get the result.

**Requirements/steps**:
- Make sure you are using placeholder syntax! You may find the documentation, linked above, useful!
- The test _will_ contain some malicious values for the `level` parameter.
- Do not perform any data manipulations in Pandas. You can use `pd.read_sql(query, conn, params=params)` to get the result.

In [36]:
### Solution - Exercise 9  
def get_jobs_by_level_safe(level, conn):
    query = "SELECT * FROM job_titles WHERE career_level = ?"
    jobs_by_level_result = pd.read_sql(query, conn, params=(level,))
    return(jobs_by_level_result, query)
    
    
### Demo function call
demo_level = 'Entry-Level'
demo_result_get_jobs_by_level_safe, demo_get_jobs_by_level_query = get_jobs_by_level_safe(demo_level, conn)
### This example of `pd.read_sql` passes a _tuple_ containing `demo_level` as the query params
demo_query_result = pd.read_sql(demo_get_jobs_by_level_query, conn, params=(demo_level,))
print(f'{demo_result_get_jobs_by_level_safe.equals(demo_query_result)}')
display(demo_result_get_jobs_by_level_safe.head())

True


,job_title_id,business_title,civil_service_title,title_code_no,title_classification,career_level,job_category,level
0,9,CARETAKER X (HA),CARETAKER (HA),90645,Labor-3,Entry-Level,Building Operations & Maintenance,00
1,18,UNIT CLERK,CLERICAL ASSOCIATE,10251,Competitive-1,Entry-Level,Administration & Human Resources Social Services,03
2,27,Asbestos Enforcement Inspector,ASSOCIATE CHEMIST,21822,Competitive-1,Entry-Level,"Policy, Research & Analysis Public Safety, Ins...",01
3,36,CITY PEST CONTROL AIDE,CITY PEST CONTROL AIDE,90643,Labor-3,Entry-Level,Health Building Operations & Maintenance Publi...,00
4,54,HEAVY-DUTY CLEANINGS SERVICES COORDINATOR (HDCSC),COMMUNITY ASSOCIATE,56057,Non-Competitive-5,Entry-Level,Social Services,00


 

**The demo should display this output.**  

<div style="overflow-x:auto;">
    <style>
        table {
            border-collapse: collapse;
            margin: 15px 0;
            font-size: 0.9em;
            font-family: sans-serif;
            width: auto;
        }
        th, td {
            padding: 8px;
            text-align: right;
            border-bottom: 1px solid #ddd;
        }
        tr:hover {background-color: #f5f5f5;}
    </style>
    <table border="0" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>job_title_id</th>
      <th>business_title</th>
      <th>civil_service_title</th>
      <th>title_code_no</th>
      <th>title_classification</th>
      <th>career_level</th>
      <th>job_category</th>
      <th>level</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>0</th>
      <td>9</td>
      <td>CARETAKER X (HA)</td>
      <td>CARETAKER (HA)</td>
      <td>90645</td>
      <td>Labor-3</td>
      <td>Entry-Level</td>
      <td>Building Operations &amp; Maintenance</td>
      <td>00</td>
    </tr>
    <tr>
      <th>1</th>
      <td>18</td>
      <td>UNIT CLERK</td>
      <td>CLERICAL ASSOCIATE</td>
      <td>10251</td>
      <td>Competitive-1</td>
      <td>Entry-Level</td>
      <td>Administration &amp; Human Resources Social Services</td>
      <td>03</td>
    </tr>
    <tr>
      <th>2</th>
      <td>27</td>
      <td>Asbestos Enforcement Inspector</td>
      <td>ASSOCIATE CHEMIST</td>
      <td>21822</td>
      <td>Competitive-1</td>
      <td>Entry-Level</td>
      <td>Policy, Research &amp; Analysis Public Safety, Inspections, &amp; Enforcement</td>
      <td>01</td>
    </tr>
    <tr>
      <th>3</th>
      <td>36</td>
      <td>CITY PEST CONTROL AIDE</td>
      <td>CITY PEST CONTROL AIDE</td>
      <td>90643</td>
      <td>Labor-3</td>
      <td>Entry-Level</td>
      <td>Health Building Operations &amp; Maintenance Public Safety, Inspections, &amp; Enforcement</td>
      <td>00</td>
    </tr>
    <tr>
      <th>4</th>
      <td>54</td>
      <td>HEAVY-DUTY CLEANINGS SERVICES COORDINATOR (HDCSC)</td>
      <td>COMMUNITY ASSOCIATE</td>
      <td>56057</td>
      <td>Non-Competitive-5</td>
      <td>Entry-Level</td>
      <td>Social Services</td>
      <td>00</td>
    </tr>
  </tbody>
</table>
    </div>

**The demo should display this printed output.**
```
True
```


 ---
 <!-- Test Cell Boilerplate -->  
The cell below will test your solution for get_jobs_by_level_safe (exercise 9). The testing variables will be available for debugging under the following names in a dictionary format.  
- `input_vars` - Input variables for your solution.   
- `original_input_vars` - Copy of input variables from prior to running your solution. Any `key:value` pair in `original_input_vars` should also exist in `input_vars` - otherwise the inputs were modified by your solution.  
- `returned_output_vars` - Outputs returned by your solution.  
- `true_output_vars` - The expected output. This _should_ "match" `returned_output_vars` based on the question requirements - otherwise, your solution is not returning the correct output. 


In [37]:
### Test Cell - Exercise 9  


from cse6040_devkit.tester_fw.testers import Tester
from yaml import safe_load
from time import time

tracemalloc.start()
mem_start, peak_start = tracemalloc.get_traced_memory()
print(f"initial memory usage: {mem_start/1024/1024:.2f} MB")

# Load testing utility
with open('resource/asnlib/publicdata/execute_tests', 'rb') as f:
    executor = dill.load(f)

@run_with_timeout(error_threshold=200.0, warning_threshold=100.0)
@suppress_stdout
def execute_tests(**kwargs):
    return executor(**kwargs)


# Execute test
start_time = time()
passed, test_case_vars, e = execute_tests(func=plugins.sql_to_df_plugin(get_jobs_by_level_safe),
              ex_name='get_jobs_by_level_safe',
              key=b'btJ0h55N40azEpSYp7KWZGFYuc4IQesVgtsBQFmcS-8=', 
              n_iter=10)
# Assign test case vars for debugging
input_vars, original_input_vars, returned_output_vars, true_output_vars = test_case_vars
duration = time() - start_time
print(f"Test duration: {duration:.2f} seconds")
current_memory, peak_memory = tracemalloc.get_traced_memory()
print(f"memory after test: {current_memory/1024/1024:.2f} MB")
print(f"memory peak during test: {peak_memory/1024/1024:.2f} MB")
tracemalloc.stop()
if e: raise e
assert passed, 'The solution to get_jobs_by_level_safe did not pass the test.'

###
### AUTOGRADER TEST - DO NOT REMOVE
###

print('Passed! Please submit.')

initial memory usage: 0.00 MB
Test duration: 0.32 seconds
memory after test: 0.07 MB
memory peak during test: 2.96 MB
Passed! Please submit.


Congratulations! You should now understand one of the most common security vulnerabilities to consider when writing applications with SQL. Please remember to sanitize your user inputs when writing applications so that you don't fall victim to Bobby Tables!

## An Applied Example: Finding the Top N Cases per Group
In this section, you will use what you have learned to answer a commonly asked data analytics question: the "top n-cases per group" problem.

To begin, consider a simple example. Imagine that we wanted to filter our records to keep only the top 200 job postings, ordered by their salary. At this point, we know we can accomplish this with the following keywords:

- `ORDER BY`
- `LIMIT`

Now, consider a seemingly related problem: what if we want to keep the top 3 job postings by salary for *each group* of title classifications, as provided by the `job_titles` table? Intuitively, you might recognize that you need to *partition* the table and rank the elements. We can do this with window functions. If we do this correctly, we can keep an arbitrary number of these records; we will generalize this by saying we want `n` records. You should recognize that this will mean we need to build the query with placeholder bindings. Finally, you may realize that you will need to filter the table based on the rankings: this means you will need to calculate the rankings *first* and then filter the table accordingly. This is a great indication that a CTE would make the query easier to write.

In the exercise below, you will attempt to write a query like this.

### Exercise 10: (1 points)
**top_internal_postings_by_classification**  

**Your task:** define `top_internal_postings_by_classification` as follows:

**Activity**: Get the Freshest, Highest Paying Job by Ranking and Filtering Internal Job Postings by Classification and Posting Date

**Inputs**: 
- `n`: The number of records to keep, per group.
- `conn`: a `sqlite` connection containing all necessary tables.

**Return**: 
- `query_result`: The result of querying the database to create a new DataFrame from the `job_postings`, `jobs`, `job_titles`, and `salaries` tables with the following columns:
    - `title_classification`: The job classification title from the `job_titles` table.
    - `business_title`: The business title of the job from the `job_titles` table.
    - `posting_updated`: The date when the job posting was last updated from the `job_postings` table.
    - `salary_range_to`: The maximum salary range for the job posting from the `salaries` table.
    - `job_rank`: The rank of the job posting within its classification based on the `posting_updated` date, salary, and job ID.
- `query`: A Python string containing a SQLite query used to obtain the result.

**Requirements/steps**:
- Use a Common Table Expression (CTE) to rank job postings within each `title_classification` based on the `posting_updated` date DESCENDING, with ties broken by `salary_range_to` DESCENDING and `job_id` ASCENDING in that order. SQLite does not allow the direct use of window functions like `RANK()` in the `WHERE` clause. Therefore, a CTE is used to first calculate the ranks and the ranks are filtered in the outer query.
- Use the `PARTITION BY` clause within the `RANK()` function to group the job postings by `title_classification` before ranking them, ensuring that the ranking is applied separately within each classification.
- Filter the results to include only job postings where `posting_type` is `'Internal'`. This is necessary to avoid duplicates that occur when both internal and external versions of a job posting exist.
- Return only the top `n` job postings per `title_classification`, ranked by their `posting_updated` date.
- Order the final results by `title_classification` and the ranking.
- Do not perform any data manipulations in Pandas. You can use `pd.read_sql(query, conn, params=params)` to get the result.

**Note**: Make sure your query is parameterized. We will pass malicious values for `n` in the tests.

**Hint**: The following SQLite concepts and functions will help: [Common Table Expression (CTE)](https://www.sqlite.org/lang_with.html) as well as `RANK` and `PARTITION BY` at the [SQLite Window Functions Page](https://www.sqlite.org/windowfunctions.html).


In [47]:
### Solution - Exercise 10  
def top_internal_postings_by_classification(n, conn):
    query = """
    WITH ranked_jobs AS(
    SELECT jt.title_classification, 
    jt.business_title, 
    jp.posting_updated, 
    s.salary_range_to, 
    RANK() OVER (PARTITION BY jt.title_classification 
    ORDER BY jp.posting_updated DESC, s.salary_range_to DESC, j.job_id ASC) AS job_rank
    FROM job_postings AS jp
    JOIN jobs AS j on j.job_id = jp.job_id
    JOIN salaries AS s on s.job_id = j.job_id
    JOIN job_titles AS jt on jt.job_title_id = j.job_title_id
    WHERE jp.posting_type ='Internal')
    
    SELECT * 
    FROM ranked_jobs
    WHERE job_rank <= ?
    ORDER BY title_classification ASC, job_rank ASC
    
    """
    result = pd.read_sql(query, conn, params=(n,))
    return(result, query)
    
    
### Demo function call
demo_n = 3
demo_result_top_internal_postings_by_classification, demo_query = \
    top_internal_postings_by_classification(demo_n, conn) 
demo_query_result = pd.read_sql_query(demo_query, conn, params=(demo_n,))
print(f'{demo_result_top_internal_postings_by_classification.equals(demo_query_result)}')
display(demo_result_top_internal_postings_by_classification)

True


,title_classification,business_title,posting_updated,salary_range_to,job_rank
0,Competitive-1,Traffic Analyst - OLS,2024-08-26 00:00:00.000000,96395.0,1
1,Competitive-1,Childcare Inspection Supervisor,2024-08-26 00:00:00.000000,84616.0,2
2,Competitive-1,Associate Project Manager,2024-08-26 00:00:00.000000,84401.0,3
3,Exempt-4,Assistant District Attorney Fall 2025,2024-08-21 00:00:00.000000,100000.0,1
4,Exempt-4,Assistant District Attorney - Animal Cruelty P...,2024-08-20 00:00:00.000000,175000.0,2
5,Exempt-4,Deputy Bureau Chief-Crime Strategies Bureau,2024-08-11 00:00:00.000000,199000.0,3
6,Labor-3,ELEVATOR DISPATCHER,2024-08-23 00:00:00.000000,50569.0,1
7,Labor-3,CARETAKER X,2024-08-19 00:00:00.000000,50569.0,2
8,Labor-3,CARETAKER P (HA),2024-08-16 00:00:00.000000,50569.0,3
9,Non-Competitive-5,General Counsel,2024-08-26 00:00:00.000000,150000.0,1


 

**The demo should display this output.**  

<div style="overflow-x:auto;">
    <style>
        table {
            border-collapse: collapse;
            margin: 15px 0;
            font-size: 0.9em;
            font-family: sans-serif;
            width: auto;
        }
        th, td {
            padding: 8px;
            text-align: right;
            border-bottom: 1px solid #ddd;
        }
        tr:hover {background-color: #f5f5f5;}
    </style>
    <table border="0" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>title_classification</th>
      <th>business_title</th>
      <th>posting_updated</th>
      <th>salary_range_to</th>
      <th>job_rank</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>0</th>
      <td>Competitive-1</td>
      <td>Traffic Analyst - OLS</td>
      <td>2024-08-26 00:00:00.000000</td>
      <td>96395.00</td>
      <td>1</td>
    </tr>
    <tr>
      <th>1</th>
      <td>Competitive-1</td>
      <td>Childcare Inspection Supervisor</td>
      <td>2024-08-26 00:00:00.000000</td>
      <td>84616.00</td>
      <td>2</td>
    </tr>
    <tr>
      <th>2</th>
      <td>Competitive-1</td>
      <td>Associate Project Manager</td>
      <td>2024-08-26 00:00:00.000000</td>
      <td>84401.00</td>
      <td>3</td>
    </tr>
    <tr>
      <th>3</th>
      <td>Exempt-4</td>
      <td>Assistant District Attorney Fall 2025</td>
      <td>2024-08-21 00:00:00.000000</td>
      <td>100000.00</td>
      <td>1</td>
    </tr>
    <tr>
      <th>4</th>
      <td>Exempt-4</td>
      <td>Assistant District Attorney - Animal Cruelty Prosecution Initiative</td>
      <td>2024-08-20 00:00:00.000000</td>
      <td>175000.00</td>
      <td>2</td>
    </tr>
  </tbody>
</table>
    </div>

**The demo should display this printed output.**
```
True
```


 ---
 <!-- Test Cell Boilerplate -->  
The cell below will test your solution for top_internal_postings_by_classification (exercise 10). The testing variables will be available for debugging under the following names in a dictionary format.  
- `input_vars` - Input variables for your solution.   
- `original_input_vars` - Copy of input variables from prior to running your solution. Any `key:value` pair in `original_input_vars` should also exist in `input_vars` - otherwise the inputs were modified by your solution.  
- `returned_output_vars` - Outputs returned by your solution.  
- `true_output_vars` - The expected output. This _should_ "match" `returned_output_vars` based on the question requirements - otherwise, your solution is not returning the correct output. 


In [48]:
### Test Cell - Exercise 10  


from cse6040_devkit.tester_fw.testers import Tester
from yaml import safe_load
from time import time

tracemalloc.start()
mem_start, peak_start = tracemalloc.get_traced_memory()
print(f"initial memory usage: {mem_start/1024/1024:.2f} MB")

# Load testing utility
with open('resource/asnlib/publicdata/execute_tests', 'rb') as f:
    executor = dill.load(f)

@run_with_timeout(error_threshold=200.0, warning_threshold=100.0)
@suppress_stdout
def execute_tests(**kwargs):
    return executor(**kwargs)


# Execute test
start_time = time()
passed, test_case_vars, e = execute_tests(func=plugins.sql_to_df_plugin(top_internal_postings_by_classification),
              ex_name='top_internal_postings_by_classification',
              key=b'btJ0h55N40azEpSYp7KWZGFYuc4IQesVgtsBQFmcS-8=', 
              n_iter=10)
# Assign test case vars for debugging
input_vars, original_input_vars, returned_output_vars, true_output_vars = test_case_vars
duration = time() - start_time
print(f"Test duration: {duration:.2f} seconds")
current_memory, peak_memory = tracemalloc.get_traced_memory()
print(f"memory after test: {current_memory/1024/1024:.2f} MB")
print(f"memory peak during test: {peak_memory/1024/1024:.2f} MB")
tracemalloc.stop()
if e: raise e
assert passed, 'The solution to top_internal_postings_by_classification did not pass the test.'

###
### AUTOGRADER TEST - DO NOT REMOVE
###

print('Passed! Please submit.')

initial memory usage: 0.00 MB
Test duration: 0.76 seconds
memory after test: 0.22 MB
memory peak during test: 75.98 MB
Passed! Please submit.
